# Notebook 2 - Neural Networks

**UKACM Autumn School: AI for Computational Mechanics**

### Where Notebook 1 left off

Notebook 1 ended with logistic regression: a linear function of the inputs, followed by a sigmoid.
That object is one neuron. This notebook builds the rest of the network around it, then asks what
the extra capacity actually buys on this dataset.

Notebook 1 also left two results on the table. Volume fraction predicts the average transverse
stiffness $E_{mean}$ with $R^2 = 0.95$, and with a $V_f^2$ term, $0.99$. Volume fraction predicts
the anisotropy $\Delta E = E_{22} - E_{33}$ with $R^2 = 0.0000$. Nothing at all.

### What you will do

1. **Part 1.** Build a neuron by hand, stack neurons into a layer, stack layers into a network,
   and watch the matrix shapes as you go.
2. **Part 2.** Do one forward pass and one backward pass with pencil-and-paper numbers, then check
   every gradient against `torch.autograd`.
3. **Part 3.** Train a network on volume fraction and diameter to predict $E_{mean}$, and compare
   it honestly against the linear and quadratic fits from Notebook 1.
4. **Part 4.** Climb a ladder of four ways to represent the same microstructure, from two numbers
   to the two-point correlation compressed by PCA to fourteen named physical descriptors, and see
   which of them can predict the anisotropy that composition could not touch.
5. **Part 5.** Feed the network the raw 64x64 image. Count the parameters. Then shuffle the pixels
   and show that the network does not notice.

Part 5 is the argument for Notebook 3.

### Table of contents

| Part | Question |
|---|---|
| 1 | What is a neuron, and what is a layer? |
| 2 | What does backpropagation actually compute? |
| 3 | Does a network beat a quadratic on $E_{mean}$? |
| 4 | Which representation of the microstructure recovers the anisotropy? |
| 5 | What does flattening an image throw away? |


In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.

import os, io, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

import torch
import torch.nn as nn
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)          # matches a free Colab CPU runtime

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("numpy", np.__version__, "| torch", torch.__version__, "| device", DEVICE)


### Loading the data

Four files are needed:

- `microstructure_labels.csv`, one row per microstructure
- `microstructures_64.npz`, the binary images at 64x64
- `microstructure_descriptors_core14_64.npz`, fourteen named descriptors per microstructure with a
  one-line meaning for each, computed from the same 64x64 images
- `microstructure_descriptors_v17_30_64.npz`, the full thirty-descriptor bank, used once for
  comparison in Part 4

The next cell downloads and extracts the dataset automatically.

In [ ]:
# --- Load data ---------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/CEMS-Lab/autumn-school/main/datasets/machine_learning/NB2_data.zip"
DATA_DIR = "."

FILES = ["microstructure_labels.csv", "microstructures_64.npz",
         "microstructure_descriptors_core14_64.npz",
         "microstructure_descriptors_v17_30_64.npz"]

def _have():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES)

if not _have() and DATA_URL:
    print("Downloading ...")
    with urllib.request.urlopen(DATA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

if not _have():
    try:
        from google.colab import files
        print("Select:", ", ".join(FILES))
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            "Put " + ", ".join(FILES) + " beside the notebook, or set DATA_URL above.")

df = pd.read_csv(os.path.join(DATA_DIR, "microstructure_labels.csv"))
df["E_mean"]  = (df.E22 + df.E33) / 2
df["dE"]      =  df.E22 - df.E33
df["sy_mean"] = (df.yield22 + df.yield33) / 2
df["vf"]      =  df.vol_frac / 100.0

# Drop the flagged outliers for everything that follows
clean = df[~df.outlier_flag].copy().reset_index(drop=True)

_img = np.load(os.path.join(DATA_DIR, "microstructures_64.npz"), allow_pickle=True)
IMAGES, IMG_KEYS = _img["images"], list(_img["keys"])
IMG_IDX = {k: i for i, k in enumerate(IMG_KEYS)}

_dsc = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_core14_64.npz"), allow_pickle=True)
DESC_ALL, DESC_KEYS = _dsc["D"], list(_dsc["keys"])
DESC_NAMES    = [str(x) for x in _dsc["names"]]
DESC_MEANINGS = [str(x) for x in _dsc["meanings"]]
DESC_IDX = {k: i for i, k in enumerate(DESC_KEYS)}

_d30 = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_v17_30_64.npz"), allow_pickle=True)
D30_ALL, D30_KEYS = _d30["D"], list(_d30["keys"])
D30_NAMES = [str(x) for x in _d30["names"]]
D30_IDX = {k: i for i, k in enumerate(D30_KEYS)}

# Align images and descriptors to the cleaned label table
X_IMG  = IMAGES[[IMG_IDX[k]  for k in clean.key]].astype(np.float32)
X_DESC = DESC_ALL[[DESC_IDX[k] for k in clean.key]].astype(np.float32)
X_D30  = D30_ALL[[D30_IDX[k]  for k in clean.key]].astype(np.float32)

print(f"{len(df)} rows, {int(df.outlier_flag.sum())} outliers dropped, {len(clean)} kept")
print(f"images     {X_IMG.shape}")
print(f"descriptors{X_DESC.shape}   ({len(DESC_NAMES)} named)   full bank {X_D30.shape}")
print(f"E_mean {clean.E_mean.min():.2f} to {clean.E_mean.max():.2f} GPa   "
      f"dE {clean.dE.min():+.2f} to {clean.dE.max():+.2f} GPa")


---

# Part 1 - From one neuron to a network

### One neuron

A neuron takes a vector of inputs, forms a weighted sum, adds a bias, and passes the result through
a nonlinear activation function:

$$\boxed{\;a \;=\; \phi(z), \qquad z \;=\; \sum_{j=1}^{d} w_j x_j + b \;=\; \mathbf{w}^\top \mathbf{x} + b\;}$$

The symbols, once, so they can be reused for the rest of the school:

| Symbol | Shape | Meaning |
|---|---|---|
| $\mathbf{x}$ | $d \times 1$ | the inputs. Here $x_1 = V_f$, dimensionless, and $x_2$ = fibre diameter in pixels |
| $\mathbf{w}$ | $d \times 1$ | one weight per input. $w_j$ has units of output per unit of $x_j$ |
| $b$ | scalar | the bias, in the units of the output |
| $z$ | scalar | the pre-activation, sometimes called the logit |
| $\phi$ | | the activation function, applied elementwise |
| $a$ | scalar | the neuron's output. In GPa if the neuron is predicting a stiffness |

Everything inside the bracket is linear regression. The only new thing is $\phi$.

### The three activations, and their derivatives

The derivative is the part that matters, because backpropagation multiplies by $\phi'(z)$ every time
a gradient passes back through a neuron. An activation whose derivative is near zero stops gradients.

| Name | $\phi(z)$ | $\phi'(z)$ | Range of $\phi$ | Largest $\phi'$ |
|---|---|---|---|---|
| sigmoid | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ | $\sigma(z)\bigl(1-\sigma(z)\bigr)$ | $(0, 1)$ | $1/4$ at $z=0$ |
| tanh | $\dfrac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$ | $1 - \tanh^2(z)$ | $(-1, 1)$ | $1$ at $z=0$ |
| ReLU | $\max(0, z)$ | $1$ if $z > 0$, else $0$ | $[0, \infty)$ | $1$, for every active unit |

Both sigmoid and tanh have derivatives that decay to zero away from the origin. A unit sitting in
that tail is **saturated**: it still produces an output, but it no longer passes a gradient back.
ReLU has derivative exactly one wherever it is active, at the cost of passing nothing at all where
it is not.

The logistic regression of Notebook 1 was a single neuron with $\phi = \sigma$.

In [ ]:
# --- Schematic: one neuron ---------------------------------------------------
fig, ax = plt.subplots(figsize=(10.5, 3.8))
ax.set_xlim(0, 10.6); ax.set_ylim(0, 4.0); ax.axis("off"); ax.grid(False)

xin  = 1.1
ynodes = [3.2, 2.2, 1.2]
lbls   = ["$x_1$", "$x_2$", "$x_d$"]
wlbls  = ["$w_1$", "$w_2$", "$w_d$"]
xsum, ysum = 4.3, 2.2
xact = 6.2
xout = 8.2

for yv, lb in zip(ynodes, lbls):
    ax.add_patch(plt.Circle((xin, yv), 0.30, fc="w", ec=C_DATA, lw=1.8, zorder=3))
    ax.text(xin, yv, lb, ha="center", va="center", fontsize=11, zorder=4)
ax.text(xin, 0.55, "inputs", ha="center", fontsize=9, color="0.35")
ax.text(xin, 1.75, "$\\vdots$", ha="center", va="center", fontsize=13)

for yv, wl in zip(ynodes, wlbls):
    ax.annotate("", xy=(xsum - 0.42, ysum), xytext=(xin + 0.32, yv),
                arrowprops=dict(arrowstyle="->", lw=1.4, color=C_DATA))
    ax.text((xin + xsum) / 2, (yv + ysum) / 2 + 0.16, wl, fontsize=10, color=C_FIT,
            ha="center")

ax.add_patch(plt.Circle((xsum, ysum), 0.42, fc="w", ec="k", lw=1.8, zorder=3))
ax.text(xsum, ysum, "$\\Sigma$", ha="center", va="center", fontsize=15, zorder=4)
ax.add_patch(plt.Circle((xsum, ysum + 1.35), 0.26, fc="w", ec=C_ALT, lw=1.6, zorder=3))
ax.text(xsum, ysum + 1.35, "$b$", ha="center", va="center", fontsize=11, zorder=4)
ax.annotate("", xy=(xsum, ysum + 0.44), xytext=(xsum, ysum + 1.09),
            arrowprops=dict(arrowstyle="->", lw=1.4, color=C_ALT))
ax.text(xsum, 0.9, "weighted sum\nplus bias", ha="center", fontsize=9, color="0.35")

ax.annotate("", xy=(xact - 0.55, ysum), xytext=(xsum + 0.44, ysum),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="k"))
ax.text((xsum + xact) / 2 - 0.05, ysum + 0.22, "$z$", ha="center", fontsize=11)

ax.add_patch(plt.Rectangle((xact - 0.55, ysum - 0.55), 1.1, 1.1, fc="w", ec=C_BAD, lw=1.8))
zz = np.linspace(-4, 4, 60)
ax.plot(xact - 0.40 + 0.80 * (zz + 4) / 8, ysum - 0.36 + 0.72 / (1 + np.exp(-zz)),
        color=C_BAD, lw=1.8)
ax.text(xact, ysum - 0.92, "$\\phi$", ha="center", fontsize=12, color=C_BAD)
ax.text(xact, 0.9, "activation", ha="center", fontsize=9, color="0.35")

ax.annotate("", xy=(xout - 0.34, ysum), xytext=(xact + 0.58, ysum),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="k"))
ax.add_patch(plt.Circle((xout, ysum), 0.32, fc="w", ec=C_ALT, lw=1.8, zorder=3))
ax.text(xout, ysum, "$a$", ha="center", va="center", fontsize=12, zorder=4)
ax.text(xout, 0.9, "output", ha="center", fontsize=9, color="0.35")

ax.text(9.0, 3.35, "$z = \\mathbf{w}^{\\top}\\mathbf{x} + b$", fontsize=13, ha="center")
ax.text(9.0, 2.85, "$a = \\phi(z)$", fontsize=13, ha="center")
plt.tight_layout(); plt.show()


**What the schematic shows.** The three blue circles on the left are the inputs, one weight
$w_j$ per edge. The edges meet at the summation node, where the bias $b$ is added as an input that is
always present. The result $z$ is a single number, passed through $\phi$ to give the output $a$.
Every arrow carries a number, and the only learnable objects are the edge labels and the bias.

In [ ]:
# --- The three activations and their derivatives ------------------------------
zz = np.linspace(-6, 6, 600)
sig = 1.0 / (1.0 + np.exp(-zz))

fns = [("sigmoid", sig,          sig * (1 - sig),        C_DATA),
       ("tanh",    np.tanh(zz),  1 - np.tanh(zz)**2,     C_FIT),
       ("ReLU",    np.maximum(0, zz), (zz > 0).astype(float), C_ALT)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for name, f, fp, col in fns:
    a1.plot(zz, f,  color=col, lw=2.2, label=name)
    a2.plot(zz, fp, color=col, lw=2.2, label=name)
a1.set_ylim(-1.3, 2.5); a1.set_xlabel("$z$"); a1.set_ylabel(r"$\phi(z)$")
a1.set_title("the activation", fontsize=10); a1.legend(fontsize=8, loc="upper left")
a2.set_ylim(-0.1, 1.15); a2.set_xlabel("$z$"); a2.set_ylabel(r"$\phi'(z)$")
a2.set_title("its derivative, which is what training multiplies by", fontsize=10)
a2.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

print(f"largest derivative:  sigmoid {(sig*(1-sig)).max():.3f}   "
      f"tanh {(1-np.tanh(zz)**2).max():.3f}   ReLU 1.000")
print(f"sigmoid derivative at z = 4:  {(sig*(1-sig))[np.argmin(np.abs(zz-4))]:.4f}")


**What the right panel shows.** The sigmoid derivative never exceeds the value printed above,
so a gradient passing back through a stack of sigmoid layers is multiplied by something smaller than
one every time. Away from the origin it is smaller still, as the printed value at $z=4$ shows. The
ReLU derivative is a step: exactly one where the unit is active, exactly zero where it is not. That
is the whole of why ReLU replaced sigmoid in hidden layers.

In [ ]:
# --- A neuron, written out in full -------------------------------------------
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def tanh(z):    return np.tanh(z)
def relu(z):    return np.maximum(0.0, z)

ACTIVATIONS = {"sigmoid": sigmoid, "tanh": tanh, "relu": relu}

def neuron(x, w, b, phi=sigmoid):
    z = np.dot(w, x) + b        # weighted sum plus bias
    return phi(z), z

x_in = np.array([0.40, 10.0])           # volume fraction 0.40, diameter 10
w_in = np.array([8.0, -0.05])
b_in = -2.0

a, z = neuron(x_in, w_in, b_in)
print(f"inputs   x = {x_in}")
print(f"weights  w = {w_in}     bias b = {b_in}")
print(f"pre-activation   z = w.x + b = {z:.4f}")
print(f"activation       a = sigmoid(z) = {a:.4f}")


### What the two parameters do

Move the sliders. The weight $w$ sets how steeply the neuron switches, and its sign sets which way
round. The bias $b$ slides the switch left and right along the input axis. Change the activation and
watch the shape, not the position.

One neuron can only produce a single monotonic step or ramp. That is the whole of its expressive
power, and it is why a network needs more than one.

In [ ]:
# --- Interactive single neuron ------------------------------------------------
xs_neuron = np.linspace(-4, 4, 400)

def show_neuron(w=1.0, b=0.0, activation="sigmoid"):
    phi = ACTIVATIONS[activation]
    z = w * xs_neuron + b
    a = phi(z)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 3.6))
    a1.plot(xs_neuron, z, color=C_DATA, lw=2)
    a1.axhline(0, color="k", lw=0.8); a1.axvline(0, color="k", lw=0.8)
    a1.set_xlabel("input  $x$"); a1.set_ylabel("$z = wx + b$")
    a1.set_ylim(-8, 8)
    a1.set_title(f"the linear part:  z = {w:.1f}x + {b:.1f}", fontsize=10)

    a2.plot(xs_neuron, a, color=C_FIT, lw=2.5)
    a2.axhline(0, color="k", lw=0.8); a2.axvline(0, color="k", lw=0.8)
    a2.set_xlabel("input  $x$"); a2.set_ylabel(r"$a = \phi(z)$")
    a2.set_ylim(-1.4, 2.2)
    a2.set_title(f"after the {activation}", fontsize=10)
    plt.tight_layout(); plt.show()

interact(show_neuron,
         w=FloatSlider(1.0, min=-5, max=5, step=0.25, continuous_update=False),
         b=FloatSlider(0.0, min=-5, max=5, step=0.25, continuous_update=False),
         activation=Dropdown(options=["sigmoid", "tanh", "relu"], value="sigmoid"));


### A layer is a matrix

Put $H$ neurons side by side, all reading the same input vector. Each has its own weight vector and
its own bias. Stack the weight vectors as rows of a matrix $W \in \mathbb{R}^{H \times d}$ and
collect the biases into $\mathbf{b} \in \mathbb{R}^{H}$:

$$\boxed{\;\mathbf{a}^{(l)} \;=\; \phi\!\left(W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}\right)\;}$$

with $\mathbf{a}^{(0)} = \mathbf{x}$. Every object in that line has a shape, and getting the shapes
right is most of the work of building a network:

| Object | Shape | Meaning |
|---|---|---|
| $\mathbf{a}^{(l-1)}$ | $n_{l-1} \times 1$ | the previous layer's output, the input to this one |
| $W^{(l)}$ | $n_l \times n_{l-1}$ | row $i$ is the weight vector of neuron $i$ |
| $\mathbf{b}^{(l)}$ | $n_l \times 1$ | one bias per neuron |
| $\mathbf{a}^{(l)}$ | $n_l \times 1$ | this layer's output, $n_l$ numbers |

$\phi$ is applied elementwise, so it does not change the shape. For a batch of $N$ samples stored as
rows of $X \in \mathbb{R}^{N \times d}$ the same layer is $A = \phi(XW^\top + \mathbf{b})$, with
$A \in \mathbb{R}^{N \times n_l}$. That is one matrix multiply, which is why this is fast on hardware
built for matrix multiplies.

In [ ]:
# --- One layer, with shapes ---------------------------------------------------
rng = np.random.default_rng(SEED)

N, d, H = 5, 2, 4                       # 5 samples, 2 inputs, 4 neurons
X_batch = rng.normal(size=(N, d))
W1 = rng.normal(size=(H, d)) * 0.5
b1 = np.zeros(H)

Z1 = X_batch @ W1.T + b1
A1 = tanh(Z1)

print(f"X   {str(X_batch.shape):10s}  N samples by d inputs")
print(f"W1  {str(W1.shape):10s}  H neurons by d inputs")
print(f"b1  {str(b1.shape):10s}  one bias per neuron")
print(f"Z1  {str(Z1.shape):10s}  = X @ W1.T + b1")
print(f"A1  {str(A1.shape):10s}  = tanh(Z1)")
print()
print("first row of A1:", np.round(A1[0], 4))


### A network is layers composed

Feed the output of one layer into the next. With two hidden layers and a scalar output:

$$\hat{y} \;=\; W_3\,\phi\bigl(W_2\,\phi(W_1\mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2\bigr) + \mathbf{b}_3$$

No activation on the last layer, because the output is a stiffness in GPa and must be free to take
any value.

The activations are what make this more than a linear model. Remove them and the whole composition
collapses to a single matrix, $W_3 W_2 W_1$, which is linear regression with extra steps. The cell
below checks that claim numerically.

In [ ]:
# --- Schematic: a small multilayer network ------------------------------------
LAYER_SIZES = [2, 4, 3, 1]
NAMES = ["input\n$\\mathbf{x}$", "hidden 1\n$\\mathbf{a}^{(1)}$",
         "hidden 2\n$\\mathbf{a}^{(2)}$", "output\n$\\hat{y}$"]
COLS = [C_DATA, C_ALT, C_ALT, C_FIT]

fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.set_xlim(-0.6, 11.6); ax.set_ylim(-1.3, 5.3); ax.axis("off"); ax.grid(False)

xs_lay = np.linspace(0.7, 7.6, len(LAYER_SIZES))
centres = []
for xc, n in zip(xs_lay, LAYER_SIZES):
    ys = 2.6 + 0.85 * (0.5 * (n - 1) - np.arange(n))
    centres.append((xc, ys))

for k in range(len(LAYER_SIZES) - 1):
    x0, y0 = centres[k]
    x1, y1 = centres[k + 1]
    for ya in y0:
        for yb in y1:
            ax.plot([x0 + 0.22, x1 - 0.22], [ya, yb], color="0.75", lw=0.7, zorder=1)

for (xc, ys), col, nm in zip(centres, COLS, NAMES):
    for yv in ys:
        ax.add_patch(plt.Circle((xc, yv), 0.22, fc="w", ec=col, lw=1.8, zorder=3))
    ax.text(xc, 4.75, nm, ha="center", va="center", fontsize=9)
    ax.text(xc, 0.75, f"{len(ys)} unit" + ("s" if len(ys) > 1 else ""),
            ha="center", fontsize=9, color=col)

for k in range(len(LAYER_SIZES) - 1):
    xm = 0.5 * (xs_lay[k] + xs_lay[k + 1])
    nin, nout = LAYER_SIZES[k], LAYER_SIZES[k + 1]
    ax.text(xm, -0.1, f"$W^{{({k+1})}}$  {nout} x {nin}", ha="center", va="center",
            fontsize=9, color=C_FIT,
            bbox=dict(fc="w", ec=C_FIT, lw=0.8, boxstyle="round,pad=0.25"))
    ax.text(xm, -0.85, f"$+\\ \\mathbf{{b}}^{{({k+1})}}$ ({nout})", ha="center",
            fontsize=8.5, color="0.35")

ax.text(9.6, 3.0, "$\\mathbf{a}^{(l)} = \\phi\\left(W^{(l)}\\mathbf{a}^{(l-1)}"
                  " + \\mathbf{b}^{(l)}\\right)$",
        fontsize=11, ha="center", va="center")
ax.text(9.6, 2.1, "$\\phi$ on the hidden layers,\nnone on the output,\nbecause a stiffness\n"
                  "may take any value",
        fontsize=8.5, ha="center", va="center", color="0.35")
plt.tight_layout(); plt.show()

print(f"{'layer':8s} {'W shape':>12s} {'b shape':>9s} {'parameters':>11s}")
tot = 0
for k in range(len(LAYER_SIZES) - 1):
    nin, nout = LAYER_SIZES[k], LAYER_SIZES[k + 1]
    p = nin * nout + nout
    tot += p
    print(f"{k+1:<8d} {f'{nout} x {nin}':>12s} {nout:>9d} {p:>11d}")
print(f"{'total':8s} {'':>12s} {'':>9s} {tot:>11d}")


**What the schematic shows.** Circles are units, grey lines are weights. Every unit in one
layer connects to every unit in the next, which is what "fully connected" means. Each orange box is
one weight matrix, labelled with its shape: rows are the units it feeds, columns are the units it
reads. The printed table counts the parameters layer by layer, which is the next equation.

### Counting the parameters of a dense layer

A dense layer from $n_{in}$ inputs to $n_{out}$ outputs holds a weight matrix of
$n_{in} \times n_{out}$ numbers and one bias per output:

$$\boxed{\;P_{\text{dense}} \;=\; n_{in}\,n_{out} + n_{out}\;}$$

The count is proportional to the product of the two sizes, so the cost of a layer is set by its
widest connection, not by its depth in the network. Part 5 uses this formula on an image and the
answer is uncomfortable.

In [ ]:
# --- Two hidden layers, and what happens without activations ------------------
W2 = rng.normal(size=(3, H)) * 0.5;  b2 = np.zeros(3)
W3 = rng.normal(size=(1, 3)) * 0.5;  b3 = np.zeros(1)

A2   = tanh(A1 @ W2.T + b2)
Yhat = A2 @ W3.T + b3

print("with activations")
for nm, M in [("X", X_batch), ("A1", A1), ("A2", A2), ("Yhat", Yhat)]:
    print(f"  {nm:5s} {M.shape}")

# same weights, activations removed
lin = ((X_batch @ W1.T + b1) @ W2.T + b2) @ W3.T + b3
W_eq = W3 @ W2 @ W1                       # the single equivalent matrix
lin_direct = X_batch @ W_eq.T + (b1 @ W2.T + b2) @ W3.T + b3

print(f"\nwithout activations, the three layers collapse to one {W_eq.shape} matrix")
print(f"  equivalent weights: {np.round(W_eq.ravel(), 5)}")
print(f"  identical outputs:  {np.allclose(lin, lin_direct)}")


In [ ]:
# --- The same network in torch ------------------------------------------------
def mlp(sizes, act=nn.ReLU):
    layers = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), act()]
    return nn.Sequential(*layers[:-1])      # drop the final activation

net = mlp([2, 4, 3, 1], act=nn.Tanh)
print(net)
print()
total = 0
for name, p in net.named_parameters():
    print(f"  {name:12s} {str(tuple(p.shape)):10s} {p.numel():5d} parameters")
    total += p.numel()
print(f"  {'total':12s} {'':10s} {total:5d}")


### How a network builds a function out of pieces

A tanh hidden unit is one soft step. A network with $H$ hidden units and a linear output computes

$$\hat{y}(x) \;=\; \sum_{k=1}^{H} w_{2,k}\,\tanh\!\left(w_{1,k}x + b_{1,k}\right) \;+\; b_2$$

so the output is a sum of $H$ soft steps, each free to choose its own position, sharpness and
height. Fitting is choosing where to put the steps.

The animation trains a network with six hidden units on a one-dimensional function, so that the
pieces can be drawn separately. Left: the target and the network output. Right: the six
contributions $w_{2,k}h_k(x)$ drawn individually, and their sum, which is the same orange curve as
on the left.

Watch the steps slide into position and change height. Nothing else is happening during training.

In [ ]:
# --- A six-unit network fitting a 1D function --------------------------------
torch.manual_seed(3)
H1D   = 6
x1d   = np.linspace(-3, 3, 201).astype(np.float32)
f1d   = (np.sin(3 * x1d) + 0.3 * x1d ** 2).astype(np.float32)
Xt1d  = torch.tensor(x1d).view(-1, 1)
Yt1d  = torch.tensor(f1d).view(-1, 1)

Wa = torch.randn(H1D, 1); ba = torch.randn(H1D)
Wb = torch.randn(1, H1D) * 0.4; bb = torch.zeros(1)
for p in (Wa, ba, Wb, bb):
    p.requires_grad_(True)
opt1d = torch.optim.Adam([Wa, ba, Wb, bb], lr=0.05)

N_STEP_1D = 1200
keep1d = set(np.unique(np.linspace(0, N_STEP_1D, 50).astype(int)).tolist())
SNAP1D = []
t0 = time.time()
for it in range(N_STEP_1D + 1):
    hh1d  = torch.tanh(Xt1d @ Wa.T + ba)
    pr1d  = hh1d @ Wb.T + bb
    ls1d  = ((pr1d - Yt1d) ** 2).mean()
    if it in keep1d:
        SNAP1D.append((it, float(ls1d), (hh1d * Wb).detach().numpy(),
                       pr1d.detach().numpy().ravel()))
    opt1d.zero_grad(); ls1d.backward(); opt1d.step()
print(f"trained {H1D} hidden units for {N_STEP_1D} steps in {time.time()-t0:.2f} s")
print(f"MSE at step 0    {SNAP1D[0][1]:.4f}")
print(f"MSE at step {N_STEP_1D} {SNAP1D[-1][1]:.4f}")

fig, (a1d, a2d) = plt.subplots(1, 2, figsize=(12.4, 4.4))
a1d.plot(x1d, f1d, color="0.55", lw=2.6, label="target function")
fit1d, = a1d.plot([], [], color=C_FIT, lw=2.2, label="network output")
a1d.set_xlim(-3, 3); a1d.set_ylim(-2.2, 4.4)
a1d.set_xlabel("$x$"); a1d.set_ylabel("$f(x)$")
a1d.legend(fontsize=9, loc="upper center"); a1d.set_title("the fit")
txt1d = a1d.text(0.02, 0.03, "", transform=a1d.transAxes, fontsize=9, va="bottom")

cols1d = plt.cm.tab10(np.linspace(0, 0.9, H1D))
piece  = [a2d.plot([], [], lw=1.6, color=cols1d[k], label=f"unit {k+1}")[0] for k in range(H1D)]
sum1d, = a2d.plot([], [], color=C_FIT, lw=2.4, label="sum + $b_2$")
a2d.plot(x1d, f1d, color="0.55", lw=1.6, ls="--", label="target")
a2d.set_xlim(-3, 3); a2d.set_ylim(-4.5, 4.8)
a2d.set_xlabel("$x$"); a2d.set_ylabel("$w_{2,k}\\,h_k(x)$")
a2d.legend(fontsize=7.5, ncol=4, loc="lower center")
a2d.set_title("the pieces the output is built from")

def update_1d(f):
    it, lv, contrib, pred = SNAP1D[f]
    fit1d.set_data(x1d, pred); sum1d.set_data(x1d, pred)
    for k in range(H1D):
        piece[k].set_data(x1d, contrib[:, k])
    txt1d.set_text(f"step {it},  MSE = {lv:.4f}")
    return []

anim_1d = animation.FuncAnimation(fig, update_1d, frames=len(SNAP1D), interval=150, blit=False)
plt.close(fig)
HTML(anim_1d.to_jshtml())


**What the animation shows.** Each coloured curve on the right is one hidden unit's
contribution. It is a step: flat, then a transition, then flat again. The unit's input weight sets
how sharp the transition is, its bias sets where it sits on the $x$ axis, and its output weight sets
how tall it is and which way up.

Training moves the steps. Early on they are all bunched near the origin and the sum is nearly flat.
By the end each step has taken responsibility for one part of the curve, and the sum tracks the
target.

Notice that the fit is visibly worse on the left of the curve than on the right. That is not a bug
and it is not bad luck with the random start. The next cell takes it apart, because the reason is
the most useful thing in this section.

In [ ]:
# --- Why the left of the curve fits worse --------------------------------------
# Three checks, each cheap. Together they identify the cause.

def fit_1d(H, seed, steps=1200, mirror=False):
    torch.manual_seed(seed)
    xx = np.linspace(-3, 3, 201).astype(np.float32)
    ff = (np.sin(3 * xx) + 0.3 * xx ** 2).astype(np.float32)
    if mirror:
        ff = ff[::-1].copy()                       # exact left-right mirror of the same target
    Xb, Yb = torch.tensor(xx).view(-1, 1), torch.tensor(ff).view(-1, 1)
    Wa_ = torch.randn(H, 1); ba_ = torch.randn(H)
    Wb_ = torch.randn(1, H) * 0.4; bb_ = torch.zeros(1)
    for q in (Wa_, ba_, Wb_, bb_):
        q.requires_grad_(True)
    op = torch.optim.Adam([Wa_, ba_, Wb_, bb_], lr=0.05)
    for _ in range(steps + 1):
        pr = torch.tanh(Xb @ Wa_.T + ba_) @ Wb_.T + bb_
        ls = ((pr - Yb) ** 2).mean()
        op.zero_grad(); ls.backward(); op.step()
    pr = (torch.tanh(Xb @ Wa_.T + ba_) @ Wb_.T + bb_).detach().numpy().ravel()
    e  = (pr - ff) ** 2
    return e[xx < -1].mean(), e[(xx >= -1) & (xx <= 1)].mean(), e[xx > 1].mean()

# How many bends does the target actually have?
xf = np.linspace(-3, 3, 200001); fv = np.sin(3 * xf) + 0.3 * xf ** 2
turn = xf[1:-1][np.diff(np.sign(np.diff(fv))) != 0]
print(f"turning points of the target on [-3, 3]: {np.round(turn, 2)}")
print(f"  {len(turn)} bends to place, {H1D} hidden units available\n")

# The two halves are not equally demanding
for lo, hi, nm in [(-3, -1, "left "), (1, 3, "right")]:
    m = (xf >= lo) & (xf <= hi)
    print(f"  {nm} half: f spans {fv[m].max() - fv[m].min():.2f}")
print()

print("CHECK 1  mirror the target left to right. Does the bad region follow it?")
print(f"  {'':22s}{'left':>9s}{'mid':>9s}{'right':>9s}")
for sd in (3, 5, 7):
    for mir in (False, True):
        L, M, R = fit_1d(H1D, sd, mirror=mir)
        tag = "mirrored" if mir else "original"
        print(f"  seed {sd}, {tag:10s}{L:9.4f}{M:9.4f}{R:9.4f}")

print("\nCHECK 2  is it undertraining? train ten times longer.")
for st in (1200, 12000):
    L, M, R = fit_1d(H1D, 3, steps=st)
    print(f"  {st:6d} steps:  left {L:.4f}   mid {M:.4f}   right {R:.4f}")

print("\nCHECK 3  is it capacity? add hidden units, same seed.")
for H in (6, 8, 10, 12):
    L, M, R = fit_1d(H, 3)
    print(f"  H = {H:2d}:  left {L:.4f}   mid {M:.4f}   right {R:.4f}")


**What the three checks show.**

The target has six turning points and the network has six tanh units. Each unit contributes about
one bend, so the network is exactly at capacity with nothing to spare.

Mean squared error is a single average over the whole interval. It does not ask for the error to be
spread evenly, so the optimiser puts the bends where they reduce the total most. The right half of
this function spans about 3.5 in $f$ and the left half about 2.1, so the right half is worth more
and wins. The left half is what gets given up.

**Check 1** settles that this is a property of the target and not of the code, the random seed or
the optimiser. Mirror the function and the badly fitted region mirrors with it, on every seed.

**Check 2** rules out undertraining. Ten times more steps leaves the left error unchanged.

**Check 3** shows what actually fixes it. Going from six units to twelve cuts the left error by more
than an order of magnitude, because now there are bends to spare and the competition disappears.

The practical lesson: when a fit is good in one region and poor in another, ask whether the model
had enough capacity to serve both, and remember that a global loss will always sacrifice the cheaper
region. If one region matters more to you than the loss thinks it does, that has to be put into the
loss, by weighting it, and is not something the optimiser will work out on your behalf.

---

# Part 2 - Backpropagation, concretely

### The loss

Every network in this notebook is trained on mean squared error, the same loss as Notebook 1. For
$N$ samples with targets $y_i$ and predictions $\hat{y}_i$,

$$\boxed{\;L \;=\; \frac{1}{N}\sum_{i=1}^{N}\left(\hat{y}_i - y_i\right)^2\;}$$

$L$ is in the square of the units of the target, so GPa$^2$ here. Training means choosing every
weight and bias to make $L$ small, and that needs $\partial L/\partial \theta$ for every parameter
$\theta$ in the network.

### The chain rule, backwards

Backpropagation is the chain rule applied to a composition of functions, evaluated from the output
backwards so that each intermediate derivative is computed once and reused.

Take the smallest network that still has every ingredient: two inputs, two tanh hidden units, one
linear output, squared error against a single target, so $N = 1$ and the mean above is just the one
term.

$$z^{(1)} = W_1\mathbf{x} + \mathbf{b}_1, \quad
\mathbf{h} = \tanh(z^{(1)}), \quad
\hat{y} = \mathbf{w}_2^\top\mathbf{h} + b_2, \quad
L = (\hat{y} - y)^2$$

The derivatives, in the order backpropagation computes them:

$$\frac{\partial L}{\partial \hat{y}} = 2(\hat{y}-y), \qquad
\frac{\partial L}{\partial \mathbf{w}_2} = \frac{\partial L}{\partial \hat{y}}\,\mathbf{h}, \qquad
\frac{\partial L}{\partial \mathbf{h}} = \frac{\partial L}{\partial \hat{y}}\,\mathbf{w}_2$$

$$\frac{\partial L}{\partial z^{(1)}} = \frac{\partial L}{\partial \mathbf{h}} \odot (1 - \mathbf{h}^2),
\qquad
\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial z^{(1)}}\,\mathbf{x}^\top, \qquad
\frac{\partial L}{\partial \mathbf{b}_1} = \frac{\partial L}{\partial z^{(1)}}$$

The factor $(1 - h^2)$ is the derivative of tanh, applied elementwise. Every other step is a product.
Written one entry at a time, for $k = 1, 2$ and $j = 1, 2$:

$$\frac{\partial L}{\partial W_{1,kj}} = \frac{\partial L}{\partial z^{(1)}_k}\, x_j,
\qquad
\frac{\partial L}{\partial z^{(1)}_k} = 2(\hat{y}-y)\, w_{2,k}\,\bigl(1 - h_k^2\bigr)$$

Read the second expression from right to left and it is the path a gradient takes: through the tanh,
then through the output weight, then into the loss.

### The numbers used below

The next two cells run that algebra on one fixed example. Everything on this list is chosen, not
computed:

$$\mathbf{x} = \begin{bmatrix} 0.40 \\ 1.00 \end{bmatrix}, \qquad y = 1.50$$

$$W_1 = \begin{bmatrix} 2.0 & -1.0 \\ -0.5 & 1.5 \end{bmatrix}, \quad
\mathbf{b}_1 = \begin{bmatrix} 0.10 \\ -0.20 \end{bmatrix}, \quad
\mathbf{w}_2 = \begin{bmatrix} 1.20 \\ -0.80 \end{bmatrix}, \quad
b_2 = 0.30$$

So the forward pass is, component by component,

$$z^{(1)}_1 = 2.0(0.40) + (-1.0)(1.00) + 0.10, \qquad
z^{(1)}_2 = -0.5(0.40) + 1.5(1.00) - 0.20$$

$$h_k = \tanh\bigl(z^{(1)}_k\bigr), \qquad
\hat{y} = 1.20\,h_1 - 0.80\,h_2 + 0.30, \qquad
L = (\hat{y} - 1.50)^2$$

Everything the two cells print is one of the quantities named above. Follow the printed numbers
against the equations line by line.

In [ ]:
# --- Forward pass, by hand ----------------------------------------------------
x  = np.array([0.40, 1.00])              # volume fraction 0.40, diameter rescaled to 1.0
y  = 1.50                                # target

W1h = np.array([[ 2.0, -1.0],
                [-0.5,  1.5]])
b1h = np.array([0.10, -0.20])
w2h = np.array([1.20, -0.80])
b2h = 0.30

z1 = W1h @ x + b1h
h  = np.tanh(z1)
yh = w2h @ h + b2h
L  = (yh - y) ** 2

print("FORWARD")
print(f"  x       = {x}")
print(f"  z1      = W1 x + b1      = {np.round(z1, 6)}")
print(f"  h       = tanh(z1)       = {np.round(h, 6)}")
print(f"  yhat    = w2 . h + b2    = {yh:.6f}")
print(f"  target  = {y}")
print(f"  L       = (yhat - y)^2   = {L:.6f}")


In [ ]:
# --- Backward pass, by hand ---------------------------------------------------
dL_dyh = 2.0 * (yh - y)
dL_dw2 = dL_dyh * h
dL_db2 = dL_dyh
dL_dh  = dL_dyh * w2h
dL_dz1 = dL_dh * (1.0 - h**2)
dL_dW1 = np.outer(dL_dz1, x)
dL_db1 = dL_dz1

print("BACKWARD")
print(f"  dL/dyhat = 2(yhat - y)        = {dL_dyh:.6f}")
print(f"  dL/dw2   = dL/dyhat * h       = {np.round(dL_dw2, 6)}")
print(f"  dL/db2   = dL/dyhat           = {dL_db2:.6f}")
print(f"  dL/dh    = dL/dyhat * w2      = {np.round(dL_dh, 6)}")
print(f"  1 - h^2  (tanh derivative)    = {np.round(1 - h**2, 6)}")
print(f"  dL/dz1   = dL/dh * (1 - h^2)  = {np.round(dL_dz1, 6)}")
print(f"  dL/dW1   = dL/dz1 outer x     =\n{np.round(dL_dW1, 6)}")
print(f"  dL/db1   = dL/dz1             = {np.round(dL_db1, 6)}")


### The same thing, by autograd

`torch.autograd` records the operations of the forward pass, then walks the record backwards
applying exactly the products written above. It is bookkeeping, not a different algorithm.

In [ ]:
# --- torch.autograd on the identical network ---------------------------------
tW1 = torch.tensor(W1h, requires_grad=True)
tb1 = torch.tensor(b1h, requires_grad=True)
tw2 = torch.tensor(w2h, requires_grad=True)
tb2 = torch.tensor(b2h, requires_grad=True)
tx  = torch.tensor(x)
ty  = torch.tensor(y)

tz1 = tW1 @ tx + tb1
th  = torch.tanh(tz1)
tyh = tw2 @ th + tb2
tL  = (tyh - ty) ** 2
tL.backward()

print(f"loss agrees:      {np.isclose(tL.item(), L)}")
print(f"dL/dW1 agrees:    {np.allclose(tW1.grad.numpy(), dL_dW1)}")
print(f"dL/db1 agrees:    {np.allclose(tb1.grad.numpy(), dL_db1)}")
print(f"dL/dw2 agrees:    {np.allclose(tw2.grad.numpy(), dL_dw2)}")
print(f"dL/db2 agrees:    {np.allclose(tb2.grad.item(), dL_db2)}")
print()
print("autograd dL/dW1 =")
print(np.round(tW1.grad.numpy(), 6))


Every number matches. That is the only claim worth making about autograd: it computes the
chain rule you would have computed, on a graph of any size, without you writing the derivative of
each layer.

The one thing to remember in practice is the gradient of the activation. For tanh and sigmoid that
factor is small whenever the unit is saturated, so gradients shrink as they pass back through deep
stacks of them. ReLU has derivative 1 wherever it is active, which is most of why ReLU took over.

### The same pass, animated

The two cells above printed the arithmetic. The animation below is the same arithmetic drawn on the
network, so that the shape of the computation is visible alongside the numbers.

It runs in two phases.

**Forward, left to right.** Node fill shows the value held at each unit, red positive and blue
negative, and the number is printed inside the node. Edge labels are the weights. Watch the signal
arrive at $\hat{y}$ and the loss appear.

**Backward, right to left.** This is the phase worth watching. Node fill now shows the gradient at
that unit, edge width shows the size of that weight's gradient, and the edge colour shows its sign.
Compare the numbers appearing in the panel on the right against the printed backward pass above.
They are the same eight numbers.

In [ ]:
# --- Animation: one forward pass and one backward pass -----------------------
# Same network, same weights, same inputs as the two hand-worked cells above.
P_ = {"x1": (0.6, 1.15), "x2": (0.6, -1.15), "h1": (3.4, 1.15), "h2": (3.4, -1.15),
      "yh": (6.2, 0.0), "L": (8.6, 0.0)}
R_ = 0.46
E1_ = [("x1", "h1", W1h[0, 0], dL_dW1[0, 0]), ("x2", "h1", W1h[0, 1], dL_dW1[0, 1]),
       ("x1", "h2", W1h[1, 0], dL_dW1[1, 0]), ("x2", "h2", W1h[1, 1], dL_dW1[1, 1])]
E2_ = [("h1", "yh", w2h[0], dL_dw2[0]), ("h2", "yh", w2h[1], dL_dw2[1])]
E3_ = [("yh", "L", None, None)]

def seg_(a, b):
    (x0, y0), (x1_, y1_) = P_[a], P_[b]
    d = np.hypot(x1_ - x0, y1_ - y0)
    ux, uy = (x1_ - x0) / d, (y1_ - y0) / d
    return x0 + ux * R_, y0 + uy * R_, x1_ - ux * R_, y1_ - uy * R_

FPS_ = 7
STAGES_ = [
    ("F0", "forward: the inputs are placed"),
    ("F1", "forward: $z^{(1)} = W_1\\mathbf{x}+\\mathbf{b}_1$, then $\\mathbf{h}=\\tanh(z^{(1)})$"),
    ("F2", "forward: $\\hat{y} = \\mathbf{w}_2^\\top\\mathbf{h} + b_2$"),
    ("F3", "forward: $L = (\\hat{y}-y)^2$"),
    ("B0", "backward: $\\partial L/\\partial\\hat{y} = 2(\\hat{y}-y)$"),
    ("B1", "backward: through $\\mathbf{w}_2$, giving $\\partial L/\\partial\\mathbf{w}_2$ and $\\partial L/\\partial\\mathbf{h}$"),
    ("B2", "backward: through the tanh, $\\partial L/\\partial z^{(1)} = \\partial L/\\partial\\mathbf{h}\\odot(1-\\mathbf{h}^2)$"),
    ("B3", "backward: into $W_1$, $\\partial L/\\partial W_{1,kj} = \\partial L/\\partial z^{(1)}_k\\, x_j$"),
]
N_FR_ = FPS_ * len(STAGES_)

LEDGER_ = [("$z^{(1)}$", f"[{z1[0]:+.4f}, {z1[1]:+.4f}]", 1, 0),
           ("$\\mathbf{h}=\\tanh(z^{(1)})$", f"[{h[0]:+.4f}, {h[1]:+.4f}]", 1, 0),
           ("$\\hat{y}$", f"{yh:+.4f}", 2, 0),
           ("$L=(\\hat{y}-y)^2$", f"{L:.4f}", 3, 0),
           ("$\\partial L/\\partial\\hat{y}$", f"{dL_dyh:+.4f}", 4, 1),
           ("$\\partial L/\\partial\\mathbf{w}_2$", f"[{dL_dw2[0]:+.4f}, {dL_dw2[1]:+.4f}]", 5, 1),
           ("$\\partial L/\\partial\\mathbf{h}$", f"[{dL_dh[0]:+.4f}, {dL_dh[1]:+.4f}]", 5, 1),
           ("$1-\\mathbf{h}^2$", f"[{1-h[0]**2:.4f}, {1-h[1]**2:.4f}]", 6, 1),
           ("$\\partial L/\\partial z^{(1)}$", f"[{dL_dz1[0]:+.4f}, {dL_dz1[1]:+.4f}]", 6, 1),
           ("$\\partial L/\\partial W_1$ row 1", f"[{dL_dW1[0,0]:+.4f}, {dL_dW1[0,1]:+.4f}]", 7, 1),
           ("$\\partial L/\\partial W_1$ row 2", f"[{dL_dW1[1,0]:+.4f}, {dL_dW1[1,1]:+.4f}]", 7, 1)]

fig = plt.figure(figsize=(13.2, 5.0))
gsp = fig.add_gridspec(1, 2, width_ratios=[2.15, 1.0], wspace=0.04)
axN = fig.add_subplot(gsp[0, 0]); axT = fig.add_subplot(gsp[0, 1])
axN.set_xlim(-0.4, 9.9); axN.set_ylim(-3.35, 2.6); axN.axis("off"); axN.grid(False)
axT.axis("off"); axT.grid(False)

VA_, VG_ = 1.25, 4.8                      # colour scales: activation, gradient
cmapA_ = plt.cm.coolwarm
def col_(v, V): return cmapA_(0.5 + 0.5 * np.clip(v / V, -1, 1))

e_line, e_lab, e_dot = {}, {}, {}
for a, b, w_, g_ in E1_ + E2_ + E3_:
    x0, y0, x1_, y1_ = seg_(a, b)
    e_line[(a, b)], = axN.plot([x0, x1_], [y0, y1_], color="0.80", lw=1.1, zorder=1,
                               solid_capstyle="round")
    e_dot[(a, b)], = axN.plot([], [], "o", ms=8, color=C_FIT, mec="w", mew=1.0, zorder=6)
    fr = 0.22 if (a, b) != ("yh", "L") else 0.5
    mx, my = x0 + fr * (x1_ - x0), y0 + fr * (y1_ - y0)
    dy_ = y1_ - y0
    off = 0.21 * np.sign(dy_) if abs(dy_) > 1e-6 else 0.21
    e_lab[(a, b)] = axN.text(mx, my + off, "", ha="center", va="center", fontsize=8.2,
                             color="0.35", zorder=7,
                             bbox=dict(fc="w", ec="none", alpha=0.85, boxstyle="round,pad=0.12"))

n_patch, n_val = {}, {}
NLAB_ = {"x1": "$x_1$", "x2": "$x_2$", "h1": "$h_1$", "h2": "$h_2$",
         "yh": "$\\hat{y}$", "L": "$L$"}
for k_, (px, py) in P_.items():
    c_ = plt.Circle((px, py), R_, fc="w", ec="0.45", lw=1.6, zorder=4)
    axN.add_patch(c_); n_patch[k_] = c_
    n_val[k_] = axN.text(px, py, "", ha="center", va="center", fontsize=9.0, zorder=5)
    axN.text(px, py + R_ + 0.26, NLAB_[k_], ha="center", va="bottom", fontsize=11)
for xc_, nm_ in [(P_["x1"][0], "input"), (P_["h1"][0], "hidden, tanh"),
                 (P_["yh"][0], "output"), (P_["L"][0], "loss")]:
    axN.text(xc_, 2.15, nm_, ha="center", fontsize=9, color="0.4")
zsub_ = {k_: axN.text(P_[k_][0], P_[k_][1] - R_ - 0.22, "", ha="center", va="top",
                      fontsize=8.2, color="0.40") for k_ in ("h1", "h2")}
ttl_ = axN.text(0.0, -2.25, "", ha="left", va="center", fontsize=11)
key_ = axN.text(0.0, -2.72, "", ha="left", va="top", fontsize=8.3, color="0.35")

axT.text(0.02, 0.97, "forward", fontsize=10.5, color=C_DATA, va="top", weight="bold")
axT.text(0.02, 0.58, "backward", fontsize=10.5, color=C_BAD, va="top", weight="bold")
led_ = []
nf_ = nb_ = 0
for nm_, vl_, st_, blk_ in LEDGER_:
    if blk_ == 0:
        ypos = 0.90 - 0.062 * nf_; nf_ += 1
    else:
        ypos = 0.51 - 0.062 * nb_; nb_ += 1
    led_.append((axT.text(0.04, ypos, "", fontsize=9, va="top"), nm_, vl_, st_))

def update_bp(f):
    s, t = f // FPS_, (f % FPS_) / (FPS_ - 1)
    tag, title = STAGES_[s]
    back = tag.startswith("B")
    ttl_.set_text(title)
    key_.set_text("edge width and colour: gradient size and sign\ninputs are grey, they have no gradient"
                  if back else
                  "node colour: activation value, red positive\nedge labels are the weights")
    for ln_ in e_line.values():
        ln_.set_color("0.80"); ln_.set_lw(1.1); ln_.set_zorder(1)
    for d_ in e_dot.values(): d_.set_data([], [])
    for k_ in n_patch:
        n_patch[k_].set_fc("w"); n_patch[k_].set_ec("0.45"); n_patch[k_].set_lw(1.6)
        n_val[k_].set_text("")
    for tx_ in e_lab.values(): tx_.set_text(""); tx_.set_color("0.35")
    for tx_ in zsub_.values(): tx_.set_text("")

    if not back:
        vis = {"x1": x[0], "x2": x[1]}
        if s > 1 or (s == 1 and t > 0.6): vis["h1"], vis["h2"] = h[0], h[1]
        if s > 2 or (s == 2 and t > 0.45): vis["yh"] = yh
        for k_, v_ in vis.items():
            n_patch[k_].set_fc(col_(v_, VA_)); n_val[k_].set_text(f"{v_:+.3f}")
        if s > 3 or (s == 3 and t > 0.45):
            n_patch["L"].set_fc("#f2e4c9"); n_val["L"].set_text(f"{L:.3f}")
        if s >= 1:
            for i_, k_ in enumerate(("h1", "h2")):
                zsub_[k_].set_text(f"$z^{{(1)}}_{i_+1}$ = {z1[i_]:+.3f}")
        for a, b, w_, g_ in E1_ + E2_:
            e_lab[(a, b)].set_text(f"{w_:+g}")
        act = {"F1": E1_, "F2": E2_, "F3": E3_}.get(tag)
        if tag == "F0":
            for k_ in ("x1", "x2"):
                n_patch[k_].set_ec(C_DATA); n_patch[k_].set_lw(2.6)
        elif act is not None:
            for a, b, w_, g_ in act:
                x0, y0, x1_, y1_ = seg_(a, b)
                ln_ = e_line[(a, b)]
                ln_.set_color(C_FIT); ln_.set_zorder(3)
                ln_.set_lw(1.2 + 1.3 * abs(w_) if w_ is not None else 2.6)
                e_dot[(a, b)].set_data([x0 + t * (x1_ - x0)], [y0 + t * (y1_ - y0)])
                e_dot[(a, b)].set_color(C_FIT)
    else:
        for i_, k_ in enumerate(("x1", "x2")):
            n_patch[k_].set_fc("0.93"); n_val[k_].set_text(f"{x[i_]:+.3f}")
        n_patch["L"].set_fc("#f2e4c9"); n_val["L"].set_text(f"{L:.3f}")
        n_patch["yh"].set_fc(col_(dL_dyh, VG_)); n_val["yh"].set_text(f"{dL_dyh:+.3f}")
        if s >= 6:
            for i_, k_ in enumerate(("h1", "h2")):
                n_patch[k_].set_fc(col_(dL_dz1[i_], VG_)); n_val[k_].set_text(f"{dL_dz1[i_]:+.3f}")
        elif s >= 5:
            for i_, k_ in enumerate(("h1", "h2")):
                n_patch[k_].set_fc(col_(dL_dh[i_], VG_)); n_val[k_].set_text(f"{dL_dh[i_]:+.3f}")
        else:
            for i_, k_ in enumerate(("h1", "h2")):
                n_patch[k_].set_fc(col_(h[i_], VA_)); n_val[k_].set_text(f"{h[i_]:+.3f}")
        gmax_ = max(np.abs(dL_dW1).max(), np.abs(dL_dw2).max())

        def gline_(edges, frac):
            for a, b, w_, g_ in edges:
                x0, y0, x1_, y1_ = seg_(a, b)
                cc = C_BAD if g_ > 0 else C_DATA
                ln_ = e_line[(a, b)]
                ln_.set_color(cc); ln_.set_zorder(3); ln_.set_lw(1.2 + 5.0 * abs(g_) / gmax_)
                e_dot[(a, b)].set_data([x1_ + frac * (x0 - x1_)], [y1_ + frac * (y0 - y1_)])
                e_dot[(a, b)].set_color(cc)
                e_lab[(a, b)].set_text(f"{g_:+.3f}"); e_lab[(a, b)].set_color(cc)

        if tag == "B0":
            x0, y0, x1_, y1_ = seg_("yh", "L")
            ln_ = e_line[("yh", "L")]
            ln_.set_color(C_BAD); ln_.set_lw(3.0); ln_.set_zorder(3)
            e_dot[("yh", "L")].set_data([x1_ + t * (x0 - x1_)], [y1_ + t * (y0 - y1_)])
            e_dot[("yh", "L")].set_color(C_BAD)
        elif tag == "B1":
            gline_(E2_, t)
        elif tag == "B2":
            gline_(E2_, 1.0)
            for k_ in ("h1", "h2"):
                n_patch[k_].set_ec(C_BAD); n_patch[k_].set_lw(2.8)
        elif tag == "B3":
            gline_(E2_, 1.0); gline_(E1_, t)

    for tx_, nm_, vl_, st_ in led_:
        tx_.set_text(f"{nm_} = {vl_}" if s >= st_ else "")
    return []

anim_bp = animation.FuncAnimation(fig, update_bp, frames=N_FR_, interval=180, blit=False)
plt.close(fig)
HTML(anim_bp.to_jshtml())


**What the animation shows.** The forward phase is the one that looks familiar: values enter
on the left, get multiplied by weights, get squashed by the tanh, and arrive as a single number that
is compared with the target.

The backward phase reuses that same graph in the opposite direction. One number leaves the loss,
$\partial L/\partial\hat{y}$, and every gradient in the network is obtained from it by
multiplication along the edges, with one factor of $1 - h_k^2$ picked up at each tanh.

The widest edge at the end carries $\partial L/\partial W_{1,12}$, the largest in magnitude of the
six weight gradients drawn. Its width is not decoration: it is how fast the loss changes with that
one weight, and it is the reason gradient descent moves that weight furthest on the next step.

Two things the picture makes concrete. Nothing is computed twice: $\partial L/\partial\hat{y}$ is
computed once and reused by every edge behind it, which is the whole efficiency argument for
backpropagation. And the gradient reaching $W_1$ has passed through the tanh factor, so a saturated
hidden unit, one with $|h_k|$ close to 1, passes almost nothing back.

---

# Part 3 - Why a network at all: fitting $E_{mean}$

Notebook 1 fitted $E_{mean}$ from volume fraction: $R^2 = 0.95$ for a straight line, $R^2 = 0.99$
once a $V_f^2$ term was added. The network now gets the same two inputs, volume fraction and
diameter, and about a thousand parameters to work with.

There is a fair question to ask before training: what could it possibly find? The curvature is one
quadratic term, and diameter is a null feature. A network is not going to invent information that
is not in the inputs.

All models below use the same 75/25 split and the same standardised inputs.

In [ ]:
# --- Split, and the Notebook 1 baselines -------------------------------------
X2 = clean[["vf", "diameter"]].values.astype(np.float32)
y_mean = clean["E_mean"].values.astype(np.float32)

Xtr, Xte, ytr, yte = train_test_split(X2, y_mean, test_size=0.25, random_state=SEED)
scaler2 = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler2.transform(Xtr).astype(np.float32), scaler2.transform(Xte).astype(np.float32)

lin_m  = LinearRegression().fit(Xtr[:, :1], ytr)
R2_LIN = r2_score(yte, lin_m.predict(Xte[:, :1]))

quad = lambda A: np.c_[A[:, 0], A[:, 0]**2]
quad_m = LinearRegression().fit(quad(Xtr), ytr)
R2_QUAD = r2_score(yte, quad_m.predict(quad(Xte)))

print(f"train {len(Xtr)}   test {len(Xte)}")
print(f"linear    in Vf            R2 = {R2_LIN:.4f}")
print(f"quadratic in Vf            R2 = {R2_QUAD:.4f}")


In [ ]:
# --- Train an MLP on (Vf, diameter) ------------------------------------------
torch.manual_seed(SEED)
net3 = mlp([2, 32, 32, 1], act=nn.ReLU)
N_PARAM_3 = sum(p.numel() for p in net3.parameters())

opt  = torch.optim.Adam(net3.parameters(), lr=0.02)
lossf = nn.MSELoss()

Xt = torch.tensor(Xtr_s);  yt = torch.tensor(ytr).view(-1, 1)
Xv = torch.tensor(Xte_s);  yv = torch.tensor(yte).view(-1, 1)

EPOCHS_3 = 600
grid_vf  = np.linspace(0.08, 0.62, 120).astype(np.float32)
grid_in  = scaler2.transform(np.c_[grid_vf, np.full_like(grid_vf, 10.0)]).astype(np.float32)
grid_t   = torch.tensor(grid_in)

hist_tr, hist_te, snaps = [], [], []
t0 = time.time()
for ep in range(EPOCHS_3):
    opt.zero_grad()
    l = lossf(net3(Xt), yt)
    l.backward(); opt.step()
    with torch.no_grad():
        hist_tr.append(l.item())
        hist_te.append(lossf(net3(Xv), yv).item())
        if ep % 10 == 0 or ep == EPOCHS_3 - 1:
            snaps.append(net3(grid_t).numpy().ravel())
elapsed3 = time.time() - t0

with torch.no_grad():
    R2_MLP_MEAN = r2_score(yte, net3(Xv).numpy().ravel())

print(f"MLP [2, 32, 32, 1], {N_PARAM_3} parameters, {EPOCHS_3} full-batch epochs")
print(f"training time: {elapsed3:.1f} s")
print()
print(f"{'model':32s} {'parameters':>11s} {'test R2':>9s}")
print(f"{'linear in Vf':32s} {2:>11d} {R2_LIN:9.4f}")
print(f"{'quadratic in Vf':32s} {3:>11d} {R2_QUAD:9.4f}")
print(f"{'MLP on Vf and diameter':32s} {N_PARAM_3:>11d} {R2_MLP_MEAN:9.4f}")


### Watch it fit

The animation shows the network output against volume fraction at fibre diameter 10, redrawn every
ten epochs, with the training and test loss beside it. The network starts as a rough piecewise
line, because ReLU units are piecewise linear, and bends into the curve.

In [ ]:
# --- Animated fit -------------------------------------------------------------
n_fr = min(60, len(snaps))
sel  = np.linspace(0, len(snaps) - 1, n_fr).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4))
ax1.scatter(Xtr[:, 0], ytr, s=6, alpha=0.18, color=C_DATA, label="training data")
(line,) = ax1.plot([], [], color=C_FIT, lw=3.0, label="MLP", zorder=4)
ax1.plot(grid_vf, quad_m.predict(quad(np.c_[grid_vf, np.zeros_like(grid_vf)])),
         color=C_ALT, lw=2.2, ls="--", label="quadratic fit", zorder=5)
ax1.set_xlim(0.08, 0.62); ax1.set_ylim(0, 16)
ax1.set_xlabel("$V_f$"); ax1.set_ylabel("$E_{mean}$ (GPa)")
ax1.legend(loc="upper left", fontsize=8)

(ltr,) = ax2.plot([], [], color=C_DATA, lw=1.8, label="train")
(lte,) = ax2.plot([], [], color=C_BAD,  lw=1.8, label="test")
ax2.set_xlim(0, EPOCHS_3); ax2.set_yscale("log"); ax2.set_ylim(1e-2, 1e2)
ax2.set_xlabel("epoch"); ax2.set_ylabel("MSE loss"); ax2.legend(fontsize=8)
ax2.set_title("loss")

def upd(f):
    k  = sel[f]
    ep = min(k * 10, EPOCHS_3 - 1)
    line.set_data(grid_vf, snaps[k])
    ax1.set_title(f"epoch {ep}", fontsize=10)
    ltr.set_data(np.arange(ep + 1), hist_tr[:ep + 1])
    lte.set_data(np.arange(ep + 1), hist_te[:ep + 1])
    return line, ltr, lte

anim3 = animation.FuncAnimation(fig, upd, frames=n_fr, interval=110, blit=False)
plt.close(fig)
HTML(anim3.to_jshtml())


### Read the table honestly

The network is level with the quadratic, give or take the third decimal place, using a few hundred
times as many parameters. It did not find anything a mechanician could not have written down.

That is the expected result, and it is worth stating as a rule. The relationship between volume
fraction and average transverse stiffness really is close to quadratic over this range. Once a
model can represent the true function, more capacity buys nothing, and you still pay for it in
training time, in tuning, and in the loss of a coefficient you could have read as physics.

Use a network when the function you need is one you cannot write down. On $E_{mean}$, you can.
On $\Delta E$, nobody can, which is Part 4.

---

# Part 4 - Four ways to describe a microstructure

Anisotropy is invisible to composition because two numbers cannot say where the fibres are. The
image can say it, but the image is 4096 numbers with no useful ordering. Between those two extremes
sit the representations that most surrogate modelling of microstructures actually uses.

## 4a. The question

There are four ways to turn this image into model input. They differ in how much human insight goes
in and how much is left to the machine.

| Rung | Representation | Who chooses the features |
|---|---|---|
| 1 | composition: volume fraction and fibre diameter | the person, and there are only two |
| 2 | the two-point correlation $S_2$, compressed by PCA | the person picks $S_2$, the machine picks the compression |
| 3 | named physical descriptors | the person, each one meaning something in micromechanics |
| 4 | features learned by a convolutional network | the machine, from the raw image |

Rungs 1 to 3 are this notebook. Rung 4 is Notebook 3.

Every rung is scored the same way: one 75/25 split, seed 0, outliers dropped, gradient boosted trees
on three targets. $E_{mean}$ is the average transverse stiffness, $\Delta E$ is the anisotropy, and
$\sigma_{y,mean}$ is the average transverse yield stress. The anisotropy is the target that
separates the rungs.

In [ ]:
# --- One scoring function, used for every rung --------------------------------
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

TARGETS = ["E_mean", "dE", "sy_mean"]
Y_ALL   = {t: clean[t].values.astype(np.float64) for t in TARGETS}

n_clean = len(clean)
idx_tr, idx_te = train_test_split(np.arange(n_clean), test_size=0.25, random_state=SEED)
print(f"{len(idx_tr)} training microstructures, {len(idx_te)} test, seed {SEED}")

LADDER = {}          # name -> (dim, {target: R2})

def score_rep(name, X, store=True):
    X = np.asarray(X, dtype=np.float64)
    out = {}
    for t in TARGETS:
        y = Y_ALL[t]
        m = HistGradientBoostingRegressor(random_state=SEED, max_iter=400)
        m.fit(X[idx_tr], y[idx_tr])
        out[t] = r2_score(y[idx_te], m.predict(X[idx_te]))
    if store:
        LADDER[name] = (X.shape[1], out)
    return out

def print_ladder():
    print(f"{'representation':30s} {'dim':>4s} " + " ".join(f"{t:>9s}" for t in TARGETS))
    for k, (d, r) in LADDER.items():
        print(f"{k:30s} {d:4d} " + " ".join(f"{r[t]:9.4f}" for t in TARGETS))

# --- Rung 1: composition ------------------------------------------------------
X_COMP = clean[["vol_frac", "diameter"]].values.astype(np.float64)
t0 = time.time()
score_rep("1. composition (Vf, d)", X_COMP)
print(f"gradient boosting on 2 features, 3 targets: {time.time() - t0:.1f} s\n")
print_ladder()


Composition predicts the average stiffness and the average yield stress well, as Notebook 1
already showed. On the anisotropy it scores below zero, meaning it does worse than predicting the
training mean. Two numbers cannot express a direction.

## 4b. The 2019 route: two-point correlation, then PCA

This is the method used by Pathan, Ponnusami, Pathan, Pitisongsawat, Erice, Petrinic and Tagarielli,
*Predictions of the mechanical properties of unidirectional fibre composites by supervised machine
learning*, Scientific Reports **9**, 13964 (2019),
[nature.com/articles/s41598-019-50144-w](https://www.nature.com/articles/s41598-019-50144-w). It
used this same dataset and these same homogenised targets.

The idea is to replace the image by a statistical function of it, then let a general purpose
compression pick the coordinates.

### The two-point correlation

Let $I(\mathbf{x})$ be the indicator field of the fibre phase, so $I = 1$ on fibre and $I = 0$ on
matrix. The periodic two-point correlation is

$$\boxed{\;S_2(\mathbf{r}) \;=\; \frac{1}{N_{pix}}\sum_{\mathbf{x}} I(\mathbf{x})\,I(\mathbf{x}+\mathbf{r})\;}$$

where the sum runs over all $N_{pix}$ pixels and $\mathbf{x}+\mathbf{r}$ wraps around the periodic
cell. $S_2(\mathbf{r})$ is the probability that a point and a second point offset by $\mathbf{r}$
are both fibre.

Two limits are worth knowing. At zero lag $S_2(\mathbf{0}) = V_f$, because a point is in the fibre
phase with probability $V_f$. At large lag the two points become statistically independent and
$S_2 \to V_f^2$.

That sum is a periodic autocorrelation, so the fast route is the Fourier transform:

$$S_2 = \frac{1}{N_{pix}}\,F^{-1}\!\bigl(|F(I)|^2\bigr)$$

$S_2$ is a full map over lag $\mathbf{r}$, so for a 64x64 image it has 4096 entries, exactly as many
numbers as the image. The compression is the next step, not this one.

In [ ]:
# --- S2 for every microstructure ----------------------------------------------
def s2_map(img):
    v = img.astype(np.float64)
    f = np.fft.rfft2(v)
    return np.fft.fftshift(np.fft.irfft2(f * np.conj(f), s=v.shape) / v.size)

t0 = time.time()
S2_ALL = np.array([s2_map(im).ravel() for im in X_IMG])
print(f"S2 for {len(S2_ALL)} microstructures at 64x64: {time.time() - t0:.2f} s")
print(f"each S2 map flattens to {S2_ALL.shape[1]} numbers, the image has {X_IMG[0].size}")

CEN = 64 // 2                                # index of zero lag after fftshift
_s0 = S2_ALL[:, CEN * 64 + CEN]
print()
print(f"zero-lag check: mean S2(0) = {_s0.mean():.4f}, mean pixel Vf = {X_IMG.mean():.4f}")
print(f"large-lag check: mean S2 at lag 32 in x = {S2_ALL[:, CEN * 64].mean():.4f}, "
      f"mean Vf^2 = {(X_IMG.mean(axis=(1, 2))**2).mean():.4f}")


In [ ]:
# --- Two microstructures, two S2 maps -----------------------------------------
# Pick a nearly isotropic and a clearly anisotropic microstructure at the same Vf and diameter.
cell40  = clean[(clean.vol_frac == 40) & (clean.diameter == 8)]
row_iso = cell40.loc[cell40.dE.abs().idxmin()]
row_ani = cell40.loc[cell40.dE.abs().idxmax()]

def _row_bits(row):
    i = clean.index.get_loc(row.name)
    return i, X_IMG[i], S2_ALL[i].reshape(64, 64)

i_iso, img_iso, S2_iso = _row_bits(row_iso)
i_ani, img_ani, S2_ani = _row_bits(row_ani)

LMAX = 32                                    # largest lag that fits in a 64x64 periodic cell
lags = np.arange(LMAX)
fig, ax = plt.subplots(2, 3, figsize=(12.5, 7.0))
for r, (row, img, S2m, lab) in enumerate([(row_iso, img_iso, S2_iso, "nearly isotropic"),
                                          (row_ani, img_ani, S2_ani, "anisotropic")]):
    ax[r, 0].imshow(img, cmap="gray", interpolation="nearest")
    ax[r, 0].set_title(f"{lab}\n$V_f$ = {row.vol_frac:.0f}%,  "
                       f"$\\Delta E$ = {row.dE:+.2f} GPa", fontsize=9)
    im = ax[r, 1].imshow(S2m, cmap="viridis", interpolation="nearest")
    ax[r, 1].set_title("$S_2(\\mathbf{r})$, zero lag at centre", fontsize=9)
    plt.colorbar(im, ax=ax[r, 1], shrink=0.85)
    sx = S2m[CEN, CEN:CEN + LMAX]        # lag along x
    sy = S2m[CEN:CEN + LMAX, CEN]        # lag along y
    ax[r, 2].plot(lags, sx, "-o", ms=3, color=C_DATA, label="along $x$")
    ax[r, 2].plot(lags, sy, "-s", ms=3, color=C_FIT,  label="along $y$")
    ax[r, 2].axhline(img.mean()**2, color="k", ls="--", lw=1, label="$V_f^2$")
    ax[r, 2].axvline(10, color=C_BAD, lw=1.0, ls=":")
    ax[r, 2].set_xlabel("lag (pixels)"); ax[r, 2].set_ylabel("$S_2$")
    ax[r, 2].set_title(f"$S_2$ at lag 10:  $x-y$ = {sx[10] - sy[10]:+.4f}", fontsize=9)
    ax[r, 2].legend(fontsize=8)
    for a in (ax[r, 0], ax[r, 1]):
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()


**What the six panels show.** Top row is a microstructure with almost no stiffness
anisotropy, bottom row one with a large one, both at the same volume fraction and fibre diameter.
The middle column is the $S_2$ map: the top one has roughly circular contours, the bottom one is
visibly stretched along one axis. The right column reads that stretch off as two profiles, and the
gap between the blue and orange curves at a lag of 10 pixels is printed in each title. That gap is
the directional signature the anisotropy responds to, and it is invisible to volume fraction because
both rows have the same one.

### Compressing $S_2$ with PCA

4096 numbers per microstructure is too many for 1102 training samples. Principal component analysis
finds an orthonormal basis $\{\mathbf{u}_k\}$ ordered so that the first components capture as much
of the variance of $S_2$ as possible, and represents each map by its first $K$ coefficients:

$$S_2^{(i)} \;\approx\; \bar{S_2} + \sum_{k=1}^{K} z^{(i)}_k \mathbf{u}_k,
\qquad z^{(i)}_k = \bigl(S_2^{(i)} - \bar{S_2}\bigr)\cdot\mathbf{u}_k$$

The coefficients $z_k$ become the model input. The 2019 paper used 50 of them.

**The PCA is fitted on the training split only.** The basis $\{\mathbf{u}_k\}$ and the mean
$\bar{S_2}$ are properties of the data, so fitting them on all 1470 microstructures would let the
test set influence the representation before the test set is ever scored. Fit on train, then
transform both.

In [ ]:
# --- PCA on S2, fitted on the training split only -----------------------------
from sklearn.decomposition import PCA

K_LIST = [5, 10, 20, 30, 50, 80]
K_MAX  = max(K_LIST)

t0 = time.time()
pca = PCA(n_components=K_MAX, svd_solver="full", random_state=SEED).fit(S2_ALL[idx_tr])
P_ALL = pca.transform(S2_ALL)
EVR   = np.cumsum(pca.explained_variance_ratio_)
t_pca = time.time() - t0

X_PCA = P_ALL[:, :50]
print(f"PCA fitted on {len(idx_tr)} training maps only, in {t_pca:.2f} s")
print(f"first component alone explains {pca.explained_variance_ratio_[0]:.4f} of the variance of S2")
print(f"50 components explain            {EVR[49]:.4f}")
print()
score_rep("2. S2 + PCA-50 (2019)", X_PCA)
print_ladder()


The anisotropy score goes from below zero to roughly two thirds, so $S_2$ genuinely carries
directional information that composition does not. That is the result the 2019 paper rests on.

Now the part worth stopping for. The 50 components reconstruct $S_2$ almost perfectly, as the
explained variance ratio printed above shows. If the number of components were the limitation, then
adding more of them would keep improving the anisotropy score. The next cell checks that directly.

In [ ]:
# --- How many components does the target actually need ------------------------
y_dE_full = Y_ALL["dE"]
sweep = []
t0 = time.time()
for k in K_LIST:
    Xk = P_ALL[:, :k]
    m  = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(Xk[idx_tr], y_dE_full[idx_tr])
    sweep.append(r2_score(y_dE_full[idx_te], m.predict(Xk[idx_te])))
print(f"component sweep: {time.time() - t0:.1f} s\n")

print(f"{'components K':>13s} {'variance of S2 explained':>26s} {'test R2 on dE':>15s}")
for k, r in zip(K_LIST, sweep):
    print(f"{k:13d} {EVR[k-1]:26.4f} {r:15.4f}")


**What the sweep shows.** Read the two right-hand columns against each other. Five
components already reconstruct over 98 percent of the variance of $S_2$ and score about 0.2 on the
anisotropy. By 20 components the reconstruction is within a fraction of a percent of perfect and the
anisotropy score is still around 0.43. Going from 50 components to 80 moves the reconstruction by a
few parts in ten thousand and the score by about a hundredth, as the table prints. The curve has
flattened.

The reason is that PCA compresses in order to reconstruct $S_2$, not in order to predict the
property. It ranks directions by how much they move $S_2$, and the directions that move $S_2$ most
are the ones that track volume fraction, which is exactly the part that carries no anisotropy.
Variance explained is not information retained for the target. They are different quantities and
the table shows them coming apart.

That is the general warning. Whenever a representation is compressed without reference to what it
will be used to predict, the compression quality tells you nothing about how well the target
survives.

## 4c. Named physical descriptors

The alternative is to choose the numbers by hand, each one for a reason. The fourteen descriptors
shipped with this notebook are a reduced subset of the course leader's descriptor bank, computed
from the same 64x64 images. Every one of them has a one-line meaning, which is the point: when the
model says a descriptor matters, you can go and look at the microstructure and see what it is
talking about.

In [ ]:
# --- The fourteen descriptors and what they mean ------------------------------
print(f"{'#':>3s} {'name':22s} {'meaning':62s} {'mean':>9s}")
for j, (nm, mn) in enumerate(zip(DESC_NAMES, DESC_MEANINGS)):
    print(f"{j+1:3d} {nm:22s} {mn:62s} {X_DESC[:, j].mean():9.4f}")
print()
print(f"shape {X_DESC.shape}, computed from the same {X_IMG.shape[1]}x{X_IMG.shape[2]} images")
print("lengths are in units of the cell side, except dt_mean which is in pixels")


In [ ]:
# --- What four of them are looking at -----------------------------------------
def periodic_centres(img):
    # Label fibres on a 3x3 tiling so fibres crossing the boundary are one object.
    H = img.shape[0]
    lbl, _ = ndimage.label(np.tile((img > 0.5).astype(np.uint8), (3, 3)))
    ids = np.unique(lbl[H:2*H, H:2*H]); ids = ids[ids > 0]
    cen = []
    for cid in ids:
        r, c = np.where(lbl == cid)
        cen.append([(c.mean() - H) % H, (r.mean() - H) % H])
    return np.array(cen)

def periodic_dt(img):
    H = img.shape[0]
    d = ndimage.distance_transform_edt(1 - np.tile((img > 0.5).astype(np.uint8), (3, 3)))
    return d[H:2*H, H:2*H]

img_d = img_ani                                  # the anisotropic one from 4b
cen   = periodic_centres(img_d)
H     = img_d.shape[0]

# periodic pairwise distances, in pixels
dxy = cen[:, None, :] - cen[None, :, :]
dxy = (dxy + H/2) % H - H/2
dist = np.sqrt((dxy**2).sum(-1)); np.fill_diagonal(dist, np.inf)
nn_sorted = np.sort(dist, axis=1)

DT = periodic_dt(img_d)
W  = 16
dens = img_d.reshape(H//W, W, H//W, W).mean(axis=(1, 3))

S2m = S2_ALL[i_ani].reshape(H, H)
sx  = S2m[CEN, CEN:CEN+LMAX]; sy = S2m[CEN:CEN+LMAX, CEN]

fig, ax = plt.subplots(1, 4, figsize=(15.5, 4.0))

ax[0].imshow(img_d, cmap="gray", interpolation="nearest")
for i in range(len(cen)):
    for j in np.argsort(dist[i])[:3]:
        v = (cen[j] - cen[i] + H/2) % H - H/2
        ax[0].plot([cen[i, 0], cen[i, 0] + v[0]], [cen[i, 1], cen[i, 1] + v[1]],
                   color=C_FIT, lw=0.8, alpha=0.8)
ax[0].plot(cen[:, 0], cen[:, 1], "o", ms=2.5, color=C_BAD)
ax[0].set_xlim(0, H); ax[0].set_ylim(H, 0)
ax[0].set_title(f"nearest-neighbour links\nnn3_mean = {nn_sorted[:, :3].mean()/H:.4f} "
                f"({nn_sorted[:, :3].mean():.1f} px)", fontsize=9)

im1 = ax[1].imshow(DT, cmap="magma", interpolation="nearest")
plt.colorbar(im1, ax=ax[1], shrink=0.8)
ax[1].set_title(f"distance transform of the matrix\ndt_mean = {DT[img_d < 0.5].mean():.2f} px", fontsize=9)

im2 = ax[2].imshow(dens, cmap="viridis", vmin=0, vmax=1, interpolation="nearest")
plt.colorbar(im2, ax=ax[2], shrink=0.8)
ax[2].set_title(f"local fibre fraction, {W}x{W} windows\nspread across windows = {dens.std():.4f}",
                fontsize=9)

ax[3].plot(lags, sx, "-o", ms=3, color=C_DATA, label="$S_2$ along $x$")
ax[3].plot(lags, sy, "-s", ms=3, color=C_FIT,  label="$S_2$ along $y$")
ax[3].fill_between(lags, sx, sy, color=C_BAD, alpha=0.20, label="the difference")
for L in (10, 20):
    ax[3].axvline(L, color="k", lw=0.8, ls=":")
ax[3].set_xlabel("lag (pixels)"); ax[3].set_ylabel("$S_2$")
ax[3].set_title(f"directional difference\ns2_diff_10 = {sx[10]-sy[10]:+.4f},  "
                f"s2_diff_20 = {sx[20]-sy[20]:+.4f}", fontsize=9)
ax[3].legend(fontsize=8)

for a in ax[:3]:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()


**What the four panels show.** All four are computed on the same microstructure, the
anisotropic one from 4b. Panel 1 draws a line from every fibre centre to its three nearest
neighbours across the periodic boundary, and the mean of those lengths is `nn3_mean`, a measure of
how tightly packed the fibres are. Panel 2 colours every matrix pixel by its distance to the nearest
fibre, so bright regions are resin-rich; its mean is `dt_mean`. Panel 3 divides the cell into windows
of 16x16 pixels and colours each by its local fibre fraction; the spread of those values is what
`local_density_std` measures, which is large when the fibres cluster. Panel 4 is the two $S_2$
profiles again with the shaded area between them, and `s2_diff_10` and `s2_diff_20` are the height
of that gap at two fixed lags.

Panels 1, 2 and 4 recompute their descriptor here, so the numbers in their titles are that
descriptor for this one microstructure. Panel 3 uses a coarser window grid than the shipped
descriptor, so it shows what is being measured rather than reproducing the value.

Each of these is one number a person chose to compute, for a reason they can state. That is the
difference from the PCA coefficients, which are 50 numbers nobody can name.

In [ ]:
# --- Rung 3: the fourteen named descriptors -----------------------------------
t0 = time.time()
score_rep("3. core 14 descriptors", X_DESC)
print(f"gradient boosting on 14 features, 3 targets: {time.time() - t0:.1f} s\n")
print_ladder()
print()
_r2 = LADDER["2. S2 + PCA-50 (2019)"][1]["dE"]
_r3 = LADDER["3. core 14 descriptors"][1]["dE"]
print(f"on dE: 14 named descriptors score {_r3:.4f} against {_r2:.4f} for 50 PCA coefficients,")
print(f"a gain of {_r3 - _r2:+.4f} with {50 - 14} fewer inputs")


Fourteen numbers chosen by someone who knows what a fibre composite is beat fifty chosen by
a variance criterion, and beat them on every one of the three targets. The named descriptors are not
better because there are more of them. There are fewer of them.

### The full bank, for comparison

The fourteen are a subset. The complete bank has thirty descriptors, adding further neighbour
statistics, fabric tensor components, radial bands and pair-orientation terms.

In [ ]:
# --- The full 30-descriptor bank ----------------------------------------------
score_rep("   full bank, 30 descriptors", X_D30)
print_ladder()
print()
_r30 = LADDER["   full bank, 30 descriptors"][1]["dE"]
print(f"the extra {X_D30.shape[1] - X_DESC.shape[1]} descriptors change the dE score by "
      f"{_r30 - _r3:+.4f}")


The extra sixteen descriptors are worth a few hundredths on the anisotropy and almost
nothing on the other two targets, as the cell printed. The course leader's manuscript uses the full
bank. This notebook uses fourteen, because every one of the fourteen can be explained in one line
and shown in a picture, and that is worth more in a lecture than the last few hundredths.

### Back to neural networks

Everything above used gradient boosted trees, so that the comparison between rungs was about the
representation and nothing else. Now hold the representation fixed at the fourteen descriptors and
change the model instead.

In [ ]:
# --- MLP on the 14 descriptors, target dE -------------------------------------
y_dE = clean["dE"].values.astype(np.float32)

Dtr, Dte = X_DESC[idx_tr].astype(np.float32), X_DESC[idx_te].astype(np.float32)
dtr, dte = y_dE[idx_tr], y_dE[idx_te]
scalerD  = StandardScaler().fit(Dtr)
Dtr_s    = scalerD.transform(Dtr).astype(np.float32)
Dte_s    = scalerD.transform(Dte).astype(np.float32)

def train_full_batch(model, Xtr_, ytr_, Xte_, yte_, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lf  = nn.MSELoss()
    A = torch.tensor(Xtr_); B = torch.tensor(ytr_).view(-1, 1)
    P = torch.tensor(Xte_)
    t0 = time.time()
    for _ in range(epochs):
        opt.zero_grad(); lf(model(A), B).backward(); opt.step()
    el = time.time() - t0
    with torch.no_grad():
        pred = model(P).numpy().ravel()
    return r2_score(yte_, pred), el

N_DESC = X_DESC.shape[1]
torch.manual_seed(SEED)
net_desc = mlp([N_DESC, 128, 64, 1], act=nn.ReLU)
N_PARAM_DESC = sum(p.numel() for p in net_desc.parameters())
R2_DESC_dE, el_desc = train_full_batch(net_desc, Dtr_s, dtr, Dte_s, dte, 600, 0.005)
print(f"MLP [{N_DESC}, 128, 64, 1], {N_PARAM_DESC} parameters, 600 epochs, {el_desc:.1f} s")
print(f"  test R2 on dE = {R2_DESC_dE:.4f}")

# baseline: the two composition inputs, same architecture family
Btr, Bte = X_COMP[idx_tr].astype(np.float32), X_COMP[idx_te].astype(np.float32)
sB = StandardScaler().fit(Btr)
torch.manual_seed(SEED)
net_base = mlp([2, 64, 64, 1], act=nn.ReLU)
R2_BASE_dE, el_base = train_full_batch(net_base, sB.transform(Btr).astype(np.float32), dtr,
                                       sB.transform(Bte).astype(np.float32), dte, 300, 0.01)
print(f"\nMLP [2, 64, 64, 1] on Vf and diameter, 300 epochs, {el_base:.1f} s")
print(f"  test R2 on dE = {R2_BASE_dE:.4f}")
print("\nSame model family, same target, same split. Only the input representation changed.")


In [ ]:
# --- The same 14 descriptors, three model classes -----------------------------
R2_RIDGE_dE = r2_score(dte, Ridge(alpha=1.0).fit(Dtr_s, dtr).predict(Dte_s))
R2_GB_dE    = LADDER["3. core 14 descriptors"][1]["dE"]

print(f"{'model on the same 14 descriptors':36s} {'test R2 on dE':>14s}")
print(f"{'ridge regression':36s} {R2_RIDGE_dE:14.4f}")
print(f"{'MLP [14, 128, 64, 1]':36s} {R2_DESC_dE:14.4f}")
print(f"{'gradient boosted trees':36s} {R2_GB_dE:14.4f}")


Read that table honestly. Ridge regression is a linear model with no hidden layers at all,
and on this representation it beats the MLP. The trees come out in front, but all three sit within
about seven hundredths of each other, against a swing of more than three quarters of an $R^2$
between representations.

That is the argument of this part in one table. Fourteen numbers chosen by a person moved the score
from below zero to around three quarters. Swapping the model that reads those numbers moves it by a
few hundredths. Representation first, model second.

### Which descriptor carried it

Permutation importance: shuffle one column of the test set, leaving everything else alone, and
measure how much $R^2$ falls. A column the model cannot do without is one whose shuffling hurts.

In [ ]:
# --- Permutation importance on the named descriptors --------------------------
rng_perm = np.random.default_rng(SEED)
t0 = time.time()
imp = np.zeros(N_DESC)
for j in range(N_DESC):
    drops = []
    for _ in range(5):
        Zp = Dte_s.copy()
        Zp[:, j] = rng_perm.permutation(Zp[:, j])
        with torch.no_grad():
            p = net_desc(torch.tensor(Zp)).numpy().ravel()
        drops.append(R2_DESC_dE - r2_score(dte, p))
    imp[j] = np.mean(drops)
print(f"permutation importance over {N_DESC} descriptors: {time.time() - t0:.1f} s")

order = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8.0, 4.4))
cols = [C_FIT if k == order[0] else C_DATA for k in order]
ax.barh(range(N_DESC)[::-1], imp[order], color=cols)
ax.set_yticks(range(N_DESC)[::-1])
ax.set_yticklabels([DESC_NAMES[k] for k in order], fontsize=8)
ax.set_xlabel("drop in test $R^2$ when this descriptor is shuffled")
ax.set_title("what the network uses to predict $\\Delta E$", fontsize=10)
plt.tight_layout(); plt.show()

for k in order[:3]:
    print(f"{DESC_NAMES[k]:20s} {imp[k]:7.4f}   {DESC_MEANINGS[k]}")
print()
_deq = X_DESC[:, DESC_NAMES.index("deq_mean")].mean() * 64
_nn3 = X_DESC[:, DESC_NAMES.index("nn3_mean")].mean() * 64
print(f"mean equivalent fibre diameter          {_deq:6.2f} pixels")
print(f"mean distance to the 3rd nearest fibre  {_nn3:6.2f} pixels")
print("The two fixed lags used by the s2_diff descriptors are 10 and 20 pixels.")


**What the bar chart shows.** The directional two-point descriptor `s2_diff_10` comes
first, with the third-nearest-neighbour distance close behind, and the cell prints the top three
with their meanings. The order is consistent with the physics: stiffness anisotropy comes from
fibres being more closely correlated along one axis than the other, and how far apart the neighbours
sit sets the scale on which that correlation is measured.

The cell also prints two lengths measured from the data. The lag of 10 pixels used by the winning
descriptor sits between the mean fibre diameter and the mean third-neighbour distance, so the model
is pointing at a physical length scale, and one you can go and measure without the model.

## 4d. The ladder so far

In [ ]:
# --- The three rungs, side by side --------------------------------------------
names = ["1. composition (Vf, d)", "2. S2 + PCA-50 (2019)", "3. core 14 descriptors"]
short = ["composition\n(2 numbers)", "$S_2$ + PCA-50\n(50 numbers)", "named descriptors\n(14 numbers)"]
cols  = [C_BAD, C_ALT, C_DATA]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.0))
for a, tgt, ttl, ylim in [(ax[0], "E_mean", "$E_{mean}$, average transverse stiffness", (0.970, 1.002)),
                          (ax[1], "dE",     "$\\Delta E$, anisotropy",                  (-0.15, 1.00))]:
    v = [LADDER[k][1][tgt] for k in names]
    a.set_ylim(*ylim)
    off = 0.035 * (ylim[1] - ylim[0])
    b = a.bar(range(3), v, color=cols, width=0.6)
    for r, val in zip(b, v):
        a.text(r.get_x() + r.get_width()/2, max(val, 0.0) + off, f"{val:.4f}",
               ha="center", fontsize=9)
    a.axhline(0, color="k", lw=0.8)
    a.set_xticks(range(3)); a.set_xticklabels(short, fontsize=8.5)
    a.set_ylabel("test $R^2$"); a.set_title(ttl, fontsize=10)
plt.tight_layout(); plt.show()

print_ladder()


**What the two panels show.** Left: on the average stiffness all three rungs are within a
few thousandths of one another, so the choice of representation barely matters. Right: on the
anisotropy the same three rungs are spread across almost the whole range of $R^2$, from below zero
to near the top. Note the different vertical scales; the left axis spans a few thousandths, the
right spans one.

A representation is only good or bad relative to a target. Nothing here is a general purpose ranking
of descriptors, and the same comparison on a different property could reorder them.

Notebook 3 adds the fourth rung: a convolutional network that is given the raw image and chooses its
own features. Before that, Part 5 shows why the network in this notebook cannot be given the raw
image directly.

### Architecture explorer

The descriptor set is fixed. Change the network instead, and see how little the answer moves. Each
click retrains from scratch, so allow a second or two.

In [ ]:
# --- Interactive architecture explorer ---------------------------------------
ACTS = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}

def explore_arch(depth=2, width=32, activation="relu", epochs=120):
    torch.manual_seed(SEED)
    m = mlp([N_DESC] + [width] * depth + [1], act=ACTS[activation])
    npar = sum(p.numel() for p in m.parameters())
    r2, el = train_full_batch(m, Dtr_s, dtr, Dte_s, dte, epochs, 0.01)

    with torch.no_grad():
        pred = m(torch.tensor(Dte_s)).numpy().ravel()

    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    ax.scatter(dte, pred, s=12, alpha=0.5, color=C_DATA)
    lim = [dte.min() - 0.2, dte.max() + 0.2]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("true $\\Delta E$ (GPa)"); ax.set_ylabel("predicted $\\Delta E$ (GPa)")
    ax.set_title(f"{depth} x {width} {activation}, {npar} parameters\n"
                 f"test $R^2$ = {r2:.3f}   ({el:.1f} s to train)", fontsize=10)
    plt.tight_layout(); plt.show()

interact(explore_arch,
         depth=IntSlider(2, min=1, max=3, step=1, continuous_update=False),
         width=Dropdown(options=[8, 16, 32, 64], value=32),
         activation=Dropdown(options=["relu", "tanh", "sigmoid"], value="relu"),
         epochs=IntSlider(120, min=40, max=200, step=40, continuous_update=False));


Across this range the score moves by a few hundredths and no setting is clearly best. Depth,
width and activation are second-order here. The descriptors are doing the work, and the network is
only reading them off.

---

# Part 5 - The flattening problem

The descriptors were chosen by a person who understood the physics. The promise of deep learning is
that you should not have to do that. So hand the network the image itself.

A fully connected layer takes a vector, so the 64x64 image has to be flattened into 4096 numbers
first. Every pixel then connects to every hidden unit.

Apply the dense layer formula from Part 1, $P_{\text{dense}} = n_{in} n_{out} + n_{out}$, with
$n_{in} = 64 \times 64 = 4096$. The count is linear in the number of pixels, so the parameter budget
of the first layer is set by the image resolution and by nothing else. Double the resolution in each
direction and that layer is four times larger.

The cell below prints the count layer by layer, so you can see which one dominates.

In [ ]:
# --- Parameter counts ---------------------------------------------------------
flat_sizes = [4096, 256, 64, 1]
net_flat_demo = mlp(flat_sizes, act=nn.ReLU)
N_PARAM_FLAT = sum(p.numel() for p in net_flat_demo.parameters())

print("dense layer parameters,  n_in * n_out + n_out")
print(f"{'layer':16s} {'n_in':>7s} {'n_out':>7s} {'parameters':>12s} {'share':>7s}")
for a_, b_ in zip(flat_sizes[:-1], flat_sizes[1:]):
    p_ = a_ * b_ + b_
    print(f"{f'{a_} -> {b_}':16s} {a_:>7d} {b_:>7d} {p_:>12,d} {100*p_/N_PARAM_FLAT:>6.1f}%")
print(f"{'total':16s} {'':>7s} {'':>7s} {N_PARAM_FLAT:>12,d}")
print()

print(f"{'input representation':26s} {'inputs':>8s} {'architecture':>22s} {'parameters':>12s}")
print(f"{'14 named descriptors':26s} {N_DESC:>8d} {f'[{N_DESC}, 128, 64, 1]':>22s} {N_PARAM_DESC:>12,d}")
print(f"{'raw 64x64 image':26s} {4096:>8d} {'[4096, 256, 64, 1]':>22s} {N_PARAM_FLAT:>12,d}")
print()
print(f"ratio: {N_PARAM_FLAT / N_PARAM_DESC:.0f}x more parameters, "
      f"trained on {len(Dtr)} microstructures")


**What the breakdown shows.** Almost the whole parameter count sits in the first layer, the
one that reads the pixels, as the printed share column makes clear. The two layers after it are
almost free by comparison. Every one of those first-layer weights is tied to one absolute pixel
position.

Nothing in the architecture says that pixel 100 and pixel 101 are adjacent, or that pixel 100 and
pixel 164 are one row apart. The layer sees 4096 unrelated numbers.

Train it on $E_{mean}$ first, because that target is easy: the volume fraction is just the mean
pixel value, and a first layer of uniform weights would compute it.

In [ ]:
# --- Flattened MLP on the raw image ------------------------------------------
X_FLAT = X_IMG.reshape(len(clean), -1)

Ftr, Fte, ftr, fte = train_test_split(X_FLAT, y_mean, test_size=0.25, random_state=SEED)

def train_flat(Xtr_, ytr_, Xte_, yte_, epochs=12, bs=32, seed=SEED, return_model=False):
    torch.manual_seed(seed)
    m = mlp(flat_sizes, act=nn.ReLU)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    lf  = nn.MSELoss()
    # The target is standardised using the training split only. Notebook 3 trains the identical
    # network the identical way, so the two notebooks report the same numbers.
    mu, sd = float(ytr_.mean()), float(ytr_.std())
    A = torch.tensor(Xtr_); B = torch.tensor((ytr_ - mu) / sd).view(-1, 1); P = torch.tensor(Xte_)
    n = len(A)
    t0 = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            j = perm[i:i + bs]
            opt.zero_grad(); lf(m(A[j]), B[j]).backward(); opt.step()
    el = time.time() - t0
    with torch.no_grad():
        pred = m(P).numpy().ravel() * sd + mu
    r2 = r2_score(yte_, pred)
    return (r2, el, m, (mu, sd)) if return_model else (r2, el)

R2_FLAT_REAL, el_fr, net_flat_real, (MU_F, SD_F) = train_flat(Ftr, ftr, Fte, fte, return_model=True)
print(f"flattened MLP on real images, 12 epochs: {el_fr:.1f} s")
print(f"  test R2 on E_mean = {R2_FLAT_REAL:.4f}")


### The demonstration

Now take one fixed random permutation of the 4096 pixel positions and apply it to every image in the
dataset, training and test alike. The result is visually destroyed: no fibres, no matrix, no
geometry, just noise with the right number of white pixels.

Retrain the identical network on the shuffled images.

In [ ]:
# --- Shuffle the pixels, identically for every image -------------------------
perm_pix = np.random.default_rng(1).permutation(4096)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
for k, idx in enumerate([0, 1]):
    axes[2*k].imshow(X_IMG[idx], cmap="gray", interpolation="nearest")
    axes[2*k].set_title(f"microstructure {idx}\n$V_f$ = {X_IMG[idx].mean():.3f}", fontsize=9)
    axes[2*k+1].imshow(X_FLAT[idx][perm_pix].reshape(64, 64), cmap="gray",
                       interpolation="nearest")
    axes[2*k+1].set_title(f"same pixels, shuffled\n$V_f$ = {X_IMG[idx].mean():.3f}", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("the same fixed permutation is applied to every image in the dataset", y=1.04)
plt.tight_layout(); plt.show()


**What the four panels show.** Panels one and three are real microstructures. Panels two and
four are the same two images after the permutation. The white pixel fraction is printed above each
pair and is unchanged, because a permutation moves pixels without creating or destroying them. Every
trace of geometry is gone. Predict the retrained score before running the next cell.

In [ ]:
# --- Retrain on the shuffled images ------------------------------------------
R2_FLAT_PERM, el_fp = train_flat(Ftr[:, perm_pix], ftr, Fte[:, perm_pix], fte)
print(f"flattened MLP on shuffled images, 12 epochs: {el_fp:.1f} s")
print(f"  test R2 on E_mean = {R2_FLAT_PERM:.4f}")
print()
print(f"{'input':26s} {'test R2':>9s}")
print(f"{'real microstructures':26s} {R2_FLAT_REAL:9.4f}")
print(f"{'pixels shuffled':26s} {R2_FLAT_PERM:9.4f}")
print(f"difference: {abs(R2_FLAT_REAL - R2_FLAT_PERM):.4f}")
print()

# How large is a difference of that size? Repeat the real-image run with three more seeds.
_seeds = [1, 2, 3]
_rep = [train_flat(Ftr, ftr, Fte, fte, seed=_s)[0] for _s in _seeds]
_all = [R2_FLAT_REAL] + _rep
print(f"same run, seeds {[SEED] + _seeds}: " + ", ".join(f"{v:.4f}" for v in _all))
print(f"spread between seeds (standard deviation): {np.std(_all):.4f}")
print(f"gap between real and shuffled:             {abs(R2_FLAT_REAL - R2_FLAT_PERM):.4f}")


**What this result shows.** Read the printed table before anything else. The same
architecture, the same initialisation seed, the same optimiser, the same twelve epochs and the same
train and test split were used twice. The first run saw real microstructures. The second run saw
images whose 4096 pixels had been reordered by one fixed permutation, applied identically to every
image, leaving nothing recognisable as a fibre. The gap between the two is smaller than the spread
the same run shows when only the seed is changed, both printed above.

**Why a dense layer cannot tell the difference.** The layer computes $W\mathbf{x}$. Every input pixel
reaches every neuron through its own private weight, and no weight knows where its pixel sat. Let $P$
be the permutation matrix that shuffles the image, so the shuffled input is $P\mathbf{x}$. Then

$$ (W P^{\top})\,(P\mathbf{x}) \;=\; W\mathbf{x} $$

Relabelling the pixels is undone exactly by relabelling the weights. So for every network that fits
the real images there is a network with identical loss, identical predictions and identical training
curve on the shuffled ones, and gradient descent is as free to find one as the other. No pixel
ordering is preferred a priori, which is why training from scratch on the shuffled images costs
nothing.

**This is a statement about the architecture, not about a trained network.** A network that has
already been fitted to real images is not itself permutation invariant: its weights have been tied
to that particular pixel ordering. The next cell takes the network trained on real images and,
without retraining, feeds it the shuffled test images.

In [ ]:
# --- The same trained network, fed shuffled pixels without retraining ---------
with torch.no_grad():
    _p_real = net_flat_real(torch.tensor(Fte)).numpy().ravel() * SD_F + MU_F
    _p_shuf = net_flat_real(torch.tensor(Fte[:, perm_pix])).numpy().ravel() * SD_F + MU_F

print("network trained on real images, weights frozen")
print(f"  test R2 on the real test images       {r2_score(fte, _p_real):9.4f}")
print(f"  test R2 on the shuffled test images   {r2_score(fte, _p_shuf):9.4f}")
print(f"  mean |f(x) - f(Px)| over the test set {np.abs(_p_real - _p_shuf).mean():9.3f} GPa")
print()
print("If the trained network were permutation invariant these two scores would be identical")
print("and the mean difference would be zero. Retraining on the shuffled images recovers the")
print("score; keeping the old weights does not.")


A model that can be retrained on a bag of pixels and reach the same score has no notion of
neighbour built into it, no notion of a fibre and no notion of direction. On $E_{mean}$ that costs
nothing, because the answer is close to a pixel count and a bag of pixels still has the right count.
On anisotropy it costs everything: the next cell puts the same network on $\Delta E$.

In [ ]:
# --- The same network on the target that needs geometry ----------------------
Gtr, Gte, gtr, gte = train_test_split(X_FLAT, y_dE, test_size=0.25, random_state=SEED)
R2_FLAT_dE, el_fd = train_flat(Gtr, gtr, Gte, gte)
print(f"flattened MLP on dE, 12 epochs: {el_fd:.1f} s")
print()
print(f"{'input representation':26s} {'parameters':>12s} {'test R2 on dE':>15s}")
print(f"{'Vf and diameter':26s} {sum(p.numel() for p in net_base.parameters()):>12,d} {R2_BASE_dE:15.4f}")
print(f"{'14 named descriptors':26s} {N_PARAM_DESC:>12,d} {R2_DESC_dE:15.4f}")
print(f"{'raw 64x64, flattened':26s} {N_PARAM_FLAT:>12,d} {R2_FLAT_dE:15.4f}")


Fourteen well-chosen numbers beat four thousand raw ones, with a fraction of the
parameters and a fraction of the training time.

The comparison is like for like. Both inputs come from the same 64x64 images: the fourteen
descriptors were computed from those images and nothing else. So the raw image contains strictly
more information than the descriptor vector, and the flattened network still cannot get at it,
because it has no way to express "these pixels are near each other".

What is needed is a layer whose parameters are tied to spatial offsets rather than to absolute
positions, so that the same pattern is detected wherever it occurs, and so that shuffling the pixels
destroys the answer. That layer is convolution, and it is Notebook 3.

---

# What to take away

1. A neuron is a weighted sum, a bias, and a nonlinearity. A layer is a matrix multiply. A network
   is layers composed. Without the nonlinearity the whole stack collapses to one matrix.
2. Backpropagation is the chain rule evaluated from the output backwards. Autograd does the
   bookkeeping and nothing else. Every gradient in Part 2 matched to machine precision.
3. On $E_{mean}$ the network matched the quadratic fit and did not beat it. When a two-parameter
   model can already represent the function, extra capacity is cost without benefit.
4. Part 4 climbed three rungs of representation on one fixed split: composition, then the two-point
   correlation compressed by PCA, then fourteen named descriptors. On $E_{mean}$ all three are
   within a few thousandths. On the anisotropy they are spread from below zero to near the top. A
   representation is good or bad only relative to a target.
5. Fifty PCA components reconstruct $S_2$ to better than 0.999 of its variance and still score well
   short of fourteen named descriptors on the anisotropy, and eighty components add about a
   hundredth. PCA compresses in order to reconstruct the input, not in order to predict the output.
   Variance explained is not information retained for the target.
6. Permutation importance put the directional two-point descriptors at the top, and their fixed lags
   sit between the fibre size and the near-neighbour spacing, both printed by that cell. The model
   points at a physical length scale you can go and measure without it.
7. A fully connected network **retrained** on shuffled pixels reaches the same score as one trained
   on real microstructures, because for any fixed permutation $P$ there are weights $WP^{\top}$ that
   make the permuted network compute the same function. No pixel ordering is preferred a priori.
   A network that has already been trained is not itself permutation invariant, as the measured
   difference of about 2 GPa per prediction in Part 5 shows. Either way, flattening throws the
   geometry away before training starts.

---

# Exercises

### 1. Count the parameters
For an MLP with layer sizes `[14, 128, 64, 1]`, work out the parameter count by hand from the
weight matrix and bias shapes, then check it against `sum(p.numel() for p in net.parameters())`.
Now do the same for `[4096, 256, 64, 1]` and say which single layer dominates.

### 2. Check one gradient yourself
In Part 2, change the hidden activation from tanh to sigmoid. Write the new derivative, update the
hand-computed backward pass, and confirm autograd still agrees. What happens to the size of
`dL/dW1` when the hidden units are strongly saturated?

### 3. The simple model was better
Fit $E_{mean}$ with the quadratic in $V_f$ and with the MLP from Part 3, and report test $R^2$,
parameter count and training time for each. Which would you put into a design loop that has to run
ten thousand times, and why? Now repeat for $\Delta E$ and give the opposite answer.

### 4. Prune the descriptors
Keep only the four `s2_mean` and `s2_diff` descriptors and rescore with `score_rep`. How much of
the $R^2$ on $\Delta E$ survives? Then keep only the other ten. What does the pair of numbers tell
you about where the anisotropy information sits?

### 5. Make the permutation argument sharper
Part 5 used one fixed permutation. Instead, permute each image with a *different* random
permutation, then retrain. Predict the result before you run it, for both $E_{mean}$ and
$\Delta E$, then explain the difference.

### 6. Supervise the compression
Part 4 showed that PCA on $S_2$ saturates because it compresses to reconstruct $S_2$ rather than to
predict $\Delta E$. Replace `PCA` with `PLSRegression` from `sklearn.cross_decomposition`, which
chooses its components using the target, fit it on the training split only with the same numbers of
components, and rerun the sweep. Does the saturation move?

### 7. The full bank
`X_D30` holds all thirty descriptors and `D30_NAMES` their names. Run permutation importance on a
gradient boosted model fitted to all thirty and list the ten that matter most for $\Delta E$. How
many of them are already in the fourteen, and what does that say about the choice of subset?
